# SPY Next-Day Long/Flat/Short Signal Research

This notebook develops a leakage-aware machine-learning pipeline that converts information available at the current SPY close into a next-trading-day **Short**, **Flat**, or **Long** signal.

The workflow covers data ingestion, feature engineering, volatility-adjusted labels, rolling IC/ICIR feature screening, walk-forward validation, model comparison, and a protected final holdout. Embedded plot images have been removed from the repository copy to keep the file lightweight; running the notebook regenerates them.

> Historical research prototype only. Reported trading metrics exclude transaction costs, slippage, financing, and taxes.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas_market_calendars as mcal
from scipy.stats import spearmanr
from pandas.plotting import autocorrelation_plot
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_validate

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.base import BaseEstimator, ClassifierMixin
from pandas_datareader import data as pdr

## Execution mode

The default portfolio mode skips the exhaustive hyperparameter searches and rebuilds each tuned estimator from the best parameters recorded during the original training-only walk-forward search. It still reruns feature construction, walk-forward diagnostics, model comparison, the protected final holdout, and all core visualizations.

Set `FULL_TUNING = True` only when you intentionally want to repeat all four searches (70,810 cross-validation fits in total). That path is substantially more computationally expensive.

In [ ]:
# Recommended for reviewers and routine reproduction.
# Change to True to rerun every GridSearchCV search from scratch.
FULL_TUNING = False

RECORDED_BEST_PARAMS = {
    "logistic": {
        "model__C": 0.01,
        "model__class_weight": "balanced",
    },
    "random_forest": {
        "model__class_weight": "balanced",
        "model__max_depth": 3,
        "model__max_features": 0.5,
        "model__min_samples_leaf": 10,
        "model__min_samples_split": 10,
        "model__n_estimators": 500,
    },
    "xgboost": {
        "model__colsample_bytree": 0.7,
        "model__learning_rate": 0.05,
        "model__max_depth": 2,
        "model__min_child_weight": 10,
        "model__n_estimators": 300,
        "model__reg_alpha": 0.0,
        "model__reg_lambda": 5.0,
        "model__subsample": 0.7,
    },
    "lightgbm": {
        "model__class_weight": "balanced",
        "model__colsample_bytree": 0.7,
        "model__learning_rate": 0.03,
        "model__max_depth": 3,
        "model__min_child_samples": 50,
        "model__n_estimators": 100,
        "model__num_leaves": 7,
        "model__reg_alpha": 0.1,
        "model__reg_lambda": 5.0,
        "model__subsample": 0.7,
    },
}

execution_mode = "FULL TUNING" if FULL_TUNING else "FAST REPRODUCTION"
print(f"Execution mode: {execution_mode}")

## 1. Data ingestion

Download daily SPY prices and macro-market variables. The end date is fixed so that the published notebook corresponds to the reported holdout results.

In [ ]:
# Setting date range
start = "2022-01-01"
end = "2026-05-05"

In [ ]:
# Macro risk factors
spy = yf.download("SPY", start=start, end=end, auto_adjust=True)
vix = yf.download("^VIX", start=start, end=end, auto_adjust=True)[['Close']]
dxy = yf.download("DX-Y.NYB", start=start, end=end, auto_adjust=True)[['Close']]
tnx = yf.download("^TNX", start=start, end=end, auto_adjust=True)[['Close']]
oil = yf.download("CL=F", start=start, end=end, auto_adjust=True)[['Close']]

In [ ]:
print(spy.columns)

In [ ]:
#"Close" is the price we want
spy = spy[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
spy.rename(columns={'Close': 'Price'}, inplace=True)

vix.rename(columns={'Close': 'VIX'}, inplace=True)
dxy.rename(columns={'Close': 'DXY'}, inplace=True)
tnx.rename(columns={'Close': 'TNX'}, inplace=True)
oil.rename(columns={'Close': 'OIL'}, inplace=True)

In [ ]:
# Combine Spy data and Macro risk factors
data = spy.join([vix, dxy, tnx, oil], how='inner').copy()
data.sort_index(inplace=True)

In [ ]:
#Flatten the data
data.columns = [col[0] for col in data.columns]
data.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(data.index, data['Price'])
plt.title('SPY Price Over Time')
plt.xlabel('Date')
plt.ylabel('Price')
plt.show()

In [ ]:
data[['VIX', 'DXY', 'TNX', 'OIL']].hist(figsize=(10,8), bins=40)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(data.index, data['VIX'])
axes[0, 0].set_title('VIX Over Time')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('VIX')

axes[0, 1].plot(data.index, data['DXY'])
axes[0, 1].set_title('DXY Over Time')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('DXY')

axes[1, 0].plot(data.index, data['TNX'])
axes[1, 0].set_title('TNX Over Time')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('TNX')

axes[1, 1].plot(data.index, data['OIL'])
axes[1, 1].set_title('OIL Over Time')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('OIL')

plt.suptitle('Macro Variables Over Time', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Feature engineering

Construct lagged-return, trend, momentum, volatility, macroeconomic, yield-curve, intraday, volume, calendar, regime, and cross-market features using information available by the prediction time.

In [ ]:
# ---------- 1. Lagged returns ----------
# Existing: log_ret, ret_3day, ret_5day
# New consistent names for Rolling IC candidates
data['log_ret'] = np.log(data['Price'] / data['Price'].shift(1))
data['ret_lag_2'] = np.log(data['Price'] / data['Price'].shift(2))
data['ret_lag_3'] = np.log(data['Price'] / data['Price'].shift(3))
data['ret_lag_5'] = np.log(data['Price'] / data['Price'].shift(5))
data['ret_lag_10'] = np.log(data['Price'] / data['Price'].shift(10))

lagged_return_features = [
    'log_ret',
    'ret_lag_2',
    'ret_lag_3',
    'ret_lag_5',
    'ret_lag_10'
]

In [ ]:
# Trend indicators: SMA gaps
for window in [5, 10, 20, 50]:
    data[f'SMA_{window}'] = data['Price'].rolling(window).mean()
    data[f'SMA_gap_{window}'] = (data['Price'] - data[f'SMA_{window}']) / data[f'SMA_{window}']


trend_features = [
    'SMA_gap_5',
    'SMA_gap_10',
    'SMA_gap_20',
    'SMA_gap_50'
]

In [ ]:
# 3. Momentum indicator features
#Momentum indicators: RSI
delta = data['Price'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()

rs = avg_gain / avg_loss
data['RSI_14'] = 100 - (100 / (1 + rs))

#Momentum indicators: MACD_hist
ema12 = data['Price'].ewm(span=12, adjust=False).mean()
ema26 = data['Price'].ewm(span=26, adjust=False).mean()

macd_line = ema12 - ema26

signal_line = macd_line.ewm(span=9, adjust=False).mean()

data['MACD_hist'] = macd_line - signal_line


momentum_features = [
    'RSI_14',
    'MACD_hist'
]

In [ ]:
# 4. Volatility indicator features
# Rolling volatility based on log returns
for window in [5, 10, 20, 60]:
    data[f'Volatility_{window}'] = data['log_ret'].rolling(window).std()


volatility_features = [
    'Volatility_5',
    'Volatility_10',
    'Volatility_20',
    'Volatility_60'
]

In [ ]:
# 5. Bollinger / price position features
rolling_mean_20 = data['Price'].rolling(20).mean()
rolling_std_20 = data['Price'].rolling(20).std()

upper_band = rolling_mean_20 + 2 * rolling_std_20
lower_band = rolling_mean_20 - 2 * rolling_std_20

data['BB_width'] = (upper_band - lower_band) / rolling_mean_20
data['BB_percent_b'] = (data['Price'] - lower_band) / (upper_band - lower_band)

bollinger_features = [
    'BB_width',
    'BB_percent_b'
]

In [ ]:
# 6. Macroeconomic change features
# VIX, DXY, OIL use log returns
# TNX uses simple changes because it is already a yield level
macro_lags = [1, 2, 3, 5, 10, 20]

for lag in macro_lags:
    data[f'VIX_ret_{lag}'] = np.log(data['VIX'] / data['VIX'].shift(lag))
    data[f'DXY_ret_{lag}'] = np.log(data['DXY'] / data['DXY'].shift(lag))
    data[f'TNX_change_{lag}'] = data['TNX'] - data['TNX'].shift(lag)
    data[f'OIL_ret_{lag}'] = np.log(data['OIL'] / data['OIL'].shift(lag))

macro_change_features = (
    ['VIX']
    + [f'VIX_ret_{lag}' for lag in macro_lags]
    + [f'DXY_ret_{lag}' for lag in macro_lags]
    + [f'TNX_change_{lag}' for lag in macro_lags]
    + [f'OIL_ret_{lag}' for lag in macro_lags]
)

In [ ]:
# 7. Macroeconomic rolling normalization features
for col in ['VIX', 'DXY', 'TNX', 'OIL']:
    for window in [20, 60]:
        rolling_mean = data[col].rolling(window).mean()
        rolling_std = data[col].rolling(window).std()
        data[f'{col}_z_{window}'] = (data[col] - rolling_mean) / rolling_std


macro_zscore_features = [
    'VIX_z_20',
    'DXY_z_20',
    'TNX_z_20',
    'OIL_z_20',

    'VIX_z_60',
    'DXY_z_60',
    'TNX_z_60',
    'OIL_z_60'
]

In [ ]:
feature_cols_part1 = (
    lagged_return_features
    + trend_features
    + momentum_features
    + volatility_features
    + bollinger_features
    + macro_change_features
    + macro_zscore_features
)

In [ ]:
# Safety check: keep only features that actually exist in data
feature_cols1 = [col for col in feature_cols_part1 if col in data.columns]

print("Number of candidate features:", len(feature_cols1))
print(feature_cols1)

In [ ]:
# ---------- 8. Yield Curve Features ----------
# YC_2Y10Y = 10-year Treasury yield - 2-year Treasury yield
# DGS2  = 2-year Treasury constant maturity rate
# DGS10 = 10-year Treasury constant maturity rate
try:
    # Download 2-year and 10-year Treasury yields from FRED
    yc = pdr.DataReader(['DGS2', 'DGS10'], 'fred', start, end)

    # Align FRED business-day data to SPY trading dates
    yc = yc.reindex(data.index).ffill()

    data['DGS2'] = yc['DGS2']
    data['DGS10'] = yc['DGS10']

except Exception as e:
    print("FRED yield curve data could not be loaded.")
    print("Error:", e)
    print("If DGS2 and DGS10 already exist in data, the code below will still use them.")


# Construct yield curve features only if DGS2 and DGS10 are available
if {'DGS2', 'DGS10'}.issubset(data.columns):

    data['YC_2Y10Y'] = data['DGS10'] - data['DGS2']

    data['YC_2Y10Y_change_1'] = data['YC_2Y10Y'] - data['YC_2Y10Y'].shift(1)

    for window in [20, 60]:
        rolling_mean = data['YC_2Y10Y'].rolling(window).mean()
        rolling_std = data['YC_2Y10Y'].rolling(window).std()

        data[f'YC_2Y10Y_z_{window}'] = (
            data['YC_2Y10Y'] - rolling_mean
        ) / rolling_std

    yield_curve_features = [
        'YC_2Y10Y_change_1',
        'YC_2Y10Y_z_20',
        'YC_2Y10Y_z_60'
    ]

else:
    print("DGS2 and DGS10 are not available, so yield curve features were not created.")
    yield_curve_features = []

In [ ]:
# ---------- 9. Intraday Structure Features ----------
# These features describe the within-day price behavior of SPY.

data['intraday_ret'] = np.log(data['Price'] / data['Open'])

data['overnight_gap'] = np.log(data['Open'] / data['Price'].shift(1))

data['high_low_range'] = (data['High'] - data['Low']) / data['Price']

# Close location value:
# close_location_value close to 1 means close is near daily high.
# close_location_value close to 0 means close is near daily low.
hl_spread = data['High'] - data['Low']
data['close_location_value'] = np.where(
    hl_spread != 0,
    (data['Price'] - data['Low']) / hl_spread,
    np.nan
)

intraday_features = [
    'intraday_ret',
    'overnight_gap',
    'high_low_range',
    'close_location_value'
]

In [ ]:
# ---------- 10. Volume Indicator Features ----------
# Use relative and normalized volume instead of raw volume.

data['relative_volume_20'] = data['Volume'] / data['Volume'].rolling(20).mean()

volume_mean_20 = data['Volume'].rolling(20).mean()
volume_std_20 = data['Volume'].rolling(20).std()

data['volume_z_20'] = (data['Volume'] - volume_mean_20) / volume_std_20

# Interaction between abnormal volume and return magnitude
data['volume_ret_interaction'] = data['volume_z_20'] * data['log_ret']

volume_features = [
    'relative_volume_20',
    'volume_z_20',
    'volume_ret_interaction'
]

In [ ]:
# ---------- 11. Calendar Effect Features ----------
# Calendar effects based on actual NYSE trading calendar
# Make sure data index is DatetimeIndex and timezone-naive
data.index = pd.to_datetime(data.index).tz_localize(None)

data['is_monday'] = (data.index.dayofweek == 0).astype(int)
data['is_friday'] = (data.index.dayofweek == 4).astype(int)

# Use NYSE calendar because SPY follows the U.S. equity trading calendar
nyse = mcal.get_calendar("NYSE")

# Generate NYSE trading days covering full months around the dataset.
# This avoids incorrectly marking the last available row as month-end
# when the current month is not complete.
calendar_start = data.index.min().to_period("M").start_time
calendar_end = data.index.max().to_period("M").end_time

schedule = nyse.schedule(
    start_date=calendar_start.date(),
    end_date=calendar_end.date()
)

nyse_trading_days = pd.DatetimeIndex(schedule.index).tz_localize(None)

# First and last NYSE trading day of each month
nyse_trading_days_series = pd.Series(nyse_trading_days, index=nyse_trading_days)
nyse_month_period = nyse_trading_days.to_period("M")

first_trading_days = (
    nyse_trading_days_series
    .groupby(nyse_month_period)
    .first()
)

last_trading_days = (
    nyse_trading_days_series
    .groupby(nyse_month_period)
    .last()
)

# Map actual NYSE first/last trading days back to the SPY dataset index
data['is_month_start'] = (
    data.index.normalize().isin(first_trading_days.dt.normalize())
).astype(int)

data['is_month_end'] = (
    data.index.normalize().isin(last_trading_days.dt.normalize())
).astype(int)

calendar_features = [
    'is_monday',
    'is_friday',
    'is_month_start',
    'is_month_end'
]

print("Calendar features created:", calendar_features)

print("\nCalendar feature counts:")
display(data[calendar_features].sum().to_frame("count"))

In [ ]:
# ---------- 12. Regime Features ----------
# Regime features capture whether the current market is in a high-risk or low-risk VIX environment.
# Rolling quantiles are computed using past/current data only, so this does not use future information.

vix_roll_q75 = data['VIX'].rolling(window=252, min_periods=60).quantile(0.75)
vix_roll_q25 = data['VIX'].rolling(window=252, min_periods=60).quantile(0.25)

data['vix_regime_high'] = (data['VIX'] > vix_roll_q75).astype(int)
data['vix_regime_low'] = (data['VIX'] < vix_roll_q25).astype(int)

regime_features = [
    'vix_regime_high',
    'vix_regime_low'
]

In [ ]:
# ---------- 13 Update Candidate Feature List ----------
# Keep previous candidate features and append additional feature groups.

feature_cols_part2 = (
    feature_cols1
    + yield_curve_features
    + intraday_features
    + volume_features
    + calendar_features
    + regime_features
)

# Safety check: keep only features that actually exist in data
feature_cols2 = [
    col for col in feature_cols_part2
    if col in data.columns
]

# Remove duplicates while preserving order
feature_cols2 = list(dict.fromkeys(feature_cols2))

print("Number of candidate features after additional feature expansion:", len(feature_cols2))
print(feature_cols2)

In [ ]:
# ---------- 15. U.S. Cross-Market Style Features with Multiple Return Horizons ----------
# QQQ: Nasdaq / growth / technology
# IWM: Russell 2000 / small caps / risk appetite
# DIA: Dow Jones Industrial Average / blue-chip

us_style_tickers = {
    'QQQ': 'QQQ',
    'IWM': 'IWM',
    'DIA': 'DIA'
}

style_lags = [1, 2, 3, 5, 10]
us_style_features = []

style_feature_df = pd.DataFrame(index=data.index)

for name, ticker in us_style_tickers.items():
    try:
        temp = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)

        if isinstance(temp.columns, pd.MultiIndex):
            temp.columns = temp.columns.get_level_values(0)

        temp = temp[['Close']].copy()
        temp.columns = [f'{name}_Price']

        # Align to SPY trading dates
        temp = temp.reindex(data.index).ffill()

        price_col = f'{name}_Price'
        style_feature_df[price_col] = temp[price_col]

        for lag in style_lags:
            ret_col = f'{name}_ret_{lag}'
            style_feature_df[ret_col] = np.log(
                style_feature_df[price_col] / style_feature_df[price_col].shift(lag)
            )
            us_style_features.append(ret_col)

    except Exception as e:
        print(f"{name} data could not be loaded.")
        print("Error:", e)

# Remove accidental duplicates while preserving order
us_style_features = list(dict.fromkeys(us_style_features))

# Join all style features to data at once
data = pd.concat([data, style_feature_df], axis=1)

# Defragment data for better performance in later cells
data = data.copy()

print("U.S. style features created:", len(us_style_features))
print(us_style_features)

In [ ]:
# ---------- 16. Update Candidate Feature List with Global Market Features ----------
feature_cols_part3 = (
    feature_cols2
    + us_style_features
)

# Safety check: keep only features that actually exist in data
feature_cols3 = [
    col for col in feature_cols_part3
    if col in data.columns
]

# Remove duplicates while preserving order
feature_cols3 = list(dict.fromkeys(feature_cols3))

print("Number of candidate features after adding global market features:", len(feature_cols3))
print(feature_cols3)

## 3. Prediction target and chronological holdout

The response is the next-day log return. Before any feature selection or tuning, reserve the final 20% of observations as a protected chronological test set.

In [ ]:
# Setting next-day return target
data['next_ret'] = np.log(data['Price'].shift(-1) / data['Price'])

In [ ]:
# base + yield curve + intraday + volume + calendar + regime + us style features.
feature_cols_candidate = feature_cols3.copy()

# Safety check: keep only features that actually exist in data
feature_cols_candidate = [
    col for col in feature_cols_candidate
    if col in data.columns
]

# Remove duplicates while preserving order
feature_cols_candidate = list(dict.fromkeys(feature_cols_candidate))

print("Number of candidate features before feature selection:", len(feature_cols_candidate))
print(feature_cols_candidate)

# Final X/y will be created after the time split, label generation, and Rolling IC selection.
model_data = data[feature_cols_candidate + ['next_ret']].dropna().copy()

print("model_data shape:", model_data.shape)
print("Missing values after dropna:", model_data.isna().sum().sum())

In [ ]:
# ============================================================
# Part 2 Step 2: Chronological Final Holdout Split
# ============================================================

test_size = 0.20
split_idx = int(len(model_data) * (1 - test_size))

train_cv_data = model_data.iloc[:split_idx].copy()
test_data = model_data.iloc[split_idx:].copy()

print("========== Chronological Final Holdout Split ==========")
print(f"Full model_data size: {len(model_data)}")
print(f"Train/CV size: {len(train_cv_data)}")
print(f"Test size: {len(test_data)}")
print(f"Test share: {len(test_data) / len(model_data):.2%}")

print("\nTrain/CV date range:")
print(f"{train_cv_data.index.min()}  to  {train_cv_data.index.max()}")

print("\nFinal test date range:")
print(f"{test_data.index.min()}  to  {test_data.index.max()}")

## 4. Volatility-adjusted labels and walk-forward validation

Scale the next-day return by the 20-day volatility known at the current close. Estimate the Short and Long thresholds from the training/CV sample only, then use expanding-window validation with a one-day gap.

In [ ]:
# Part 2 Step 3: Volatility-Adjusted Label Creation
# Use a volatility estimate known at day t.
# Volatility_20 is already based on past/current returns and is available before next_ret realizes.

vol_col = "Volatility_20"

if vol_col not in train_cv_data.columns:
    raise ValueError(f"{vol_col} is not found in train_cv_data. Please check volatility feature creation.")

# Avoid division by zero or extremely tiny volatility values
eps = 1e-8

train_cv_data["next_ret_vol_adj"] = train_cv_data["next_ret"] / (train_cv_data[vol_col] + eps)
test_data["next_ret_vol_adj"] = test_data["next_ret"] / (test_data[vol_col] + eps)

# Compute thresholds using train_cv_data only
q_low = train_cv_data["next_ret_vol_adj"].quantile(0.25)
q_high = train_cv_data["next_ret_vol_adj"].quantile(0.75)

print("========== Volatility-Adjusted Label Thresholds ==========")
print(f"Volatility scale used: {vol_col}")
print(f"q_low  (Short threshold): {q_low:.6f}")
print(f"q_high (Long threshold):  {q_high:.6f}")

def create_vol_adjusted_label(vol_adj_ret, q_low, q_high):
    if vol_adj_ret <= q_low:
        return -1
    elif vol_adj_ret >= q_high:
        return 1
    else:
        return 0

train_cv_data["label"] = train_cv_data["next_ret_vol_adj"].apply(
    lambda x: create_vol_adjusted_label(x, q_low, q_high)
)

test_data["label"] = test_data["next_ret_vol_adj"].apply(
    lambda x: create_vol_adjusted_label(x, q_low, q_high)
)

print("\n========== Label Distribution: Train/CV ==========")
print(train_cv_data["label"].value_counts().sort_index())
print(train_cv_data["label"].value_counts(normalize=True).sort_index())

print("\n========== Label Distribution: Test ==========")
print(test_data["label"].value_counts().sort_index())
print(test_data["label"].value_counts(normalize=True).sort_index())

In [ ]:
# Part 2: Reusable Walk-Forward CV Pipeline
# Expanding-window walk-forward validation
# This will be used later for model tuning / validation inside train_cv_data only.
# test_data remains untouched until final evaluation.

n_splits = 5
cv_gap = 1

walk_forward_cv = TimeSeriesSplit(
    n_splits=n_splits,
    gap=cv_gap
)

def summarize_walk_forward_cv(data, cv):
    """
    Summarize walk-forward validation folds.
    
    Parameters
    ----------
    data : pd.DataFrame
        Usually train_cv_data.
    cv : sklearn.model_selection.TimeSeriesSplit
        Walk-forward CV splitter.
    
    Returns
    -------
    pd.DataFrame
        Fold-level train/validation date ranges and sample sizes.
    """
    
    fold_summary = []
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(data), start=1):
        train_fold = data.iloc[train_idx]
        val_fold = data.iloc[val_idx]
        
        fold_summary.append({
            "fold": fold,
            "train_start": train_fold.index.min(),
            "train_end": train_fold.index.max(),
            "val_start": val_fold.index.min(),
            "val_end": val_fold.index.max(),
            "train_size": len(train_fold),
            "val_size": len(val_fold)
        })
    
    return pd.DataFrame(fold_summary)


walk_forward_summary = summarize_walk_forward_cv(
    data=train_cv_data,
    cv=walk_forward_cv
)

display(walk_forward_summary)

In [ ]:
feature_cols = feature_cols_candidate.copy()

## 5. Rolling IC and feature selection

Use rolling Spearman IC/ICIR for category-aware screening, then remove redundant signals through correlation clustering and a documented final cleanup. All selection uses only the training/CV period.

In [ ]:
# Part 2 Step 4: Rolling IC Summary
target_col = "next_ret_vol_adj"
ic_window = 60
min_periods = 40

# keep naming consistent
candidate_feature_cols = [col for col in feature_cols if col in train_cv_data.columns]

def compute_feature_rolling_ic_series(data, feature_col, target_col, window=60, min_periods=40):
    """
    Compute rolling Spearman IC series for one feature against the target.
    
    Returns
    -------
    pd.Series
        Rolling IC series indexed by window end date.
    """
    temp = data[[feature_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    
    if len(temp) < min_periods:
        return pd.Series(dtype=float)

    rolling_ic_values = []
    rolling_ic_dates = []

    for end_idx in range(window, len(temp) + 1):
        window_data = temp.iloc[end_idx - window:end_idx]

        if len(window_data) < min_periods:
            continue

        x = window_data[feature_col]
        y = window_data[target_col]

        if x.nunique() <= 1 or y.nunique() <= 1:
            continue

        ic, _ = spearmanr(x, y)

        if not np.isnan(ic):
            rolling_ic_values.append(ic)
            rolling_ic_dates.append(window_data.index[-1])

    return pd.Series(rolling_ic_values, index=rolling_ic_dates, name=feature_col)


def compute_rolling_ic_summary(data, feature_cols, target_col, window=60, min_periods=40):
    """
    Compute rolling IC summary for a list of features.
    
    Returns
    -------
    rolling_ic_summary : pd.DataFrame
    rolling_ic_dict : dict
        feature -> rolling IC pd.Series
    """
    ic_results = []
    rolling_ic_dict = {}

    for feature_col in feature_cols:
        ic_series = compute_feature_rolling_ic_series(
            data=data,
            feature_col=feature_col,
            target_col=target_col,
            window=window,
            min_periods=min_periods
        )

        if ic_series.empty:
            continue

        rolling_ic_dict[feature_col] = ic_series

        mean_ic = ic_series.mean()
        abs_mean_ic = ic_series.abs().mean()
        ic_std = ic_series.std()

        ic_results.append({
            "feature": feature_col,
            "mean_IC": mean_ic,
            "abs_mean_IC": abs(mean_ic),
            "IC_std": ic_std,
            "IC_IR": mean_ic / ic_std if pd.notna(ic_std) and ic_std != 0 else np.nan,
            "abs_IC_IR": abs(mean_ic / ic_std) if pd.notna(ic_std) and ic_std != 0 else np.nan,
            "positive_IC_share": (ic_series > 0).mean(),
            "non_missing_windows": ic_series.notna().sum()
        })

    rolling_ic_summary = pd.DataFrame(ic_results)

    if not rolling_ic_summary.empty:
        rolling_ic_summary = rolling_ic_summary.sort_values(
            by=["abs_IC_IR", "abs_mean_IC"],
            ascending=False
        ).reset_index(drop=True)

    return rolling_ic_summary, rolling_ic_dict


rolling_ic_summary, rolling_ic_dict = compute_rolling_ic_summary(
    data=train_cv_data,
    feature_cols=candidate_feature_cols,
    target_col=target_col,
    window=ic_window,
    min_periods=min_periods
)

print("Rolling IC summary shape:", rolling_ic_summary.shape)
display(rolling_ic_summary.head(30))

In [ ]:
# Rolling IC Visualization 1: Top Features Summary
top_n = 20
top_ic_features = rolling_ic_summary.head(top_n).copy()

# Plot 1: Top features by abs_IC_IR
plot_df = top_ic_features.sort_values("abs_IC_IR", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(plot_df["feature"], plot_df["abs_IC_IR"])
plt.xlabel("abs_IC_IR")
plt.ylabel("Feature")
plt.title(f"Top {top_n} Features by Absolute IC Information Ratio")
plt.tight_layout()
plt.show()

# Plot 2: Top features by abs_mean_IC
plot_df = top_ic_features.sort_values("abs_mean_IC", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(plot_df["feature"], plot_df["abs_mean_IC"])
plt.xlabel("abs_mean_IC")
plt.ylabel("Feature")
plt.title(f"Top {top_n} Features by Absolute Mean Rolling IC")
plt.tight_layout()
plt.show()

In [ ]:
# Rolling IC Visualization 2: Rolling IC Time Series
top_plot_n = 20
highlight_n = 5

top_features_to_plot = rolling_ic_summary.head(top_plot_n)["feature"].tolist()
highlight_features = top_features_to_plot[:highlight_n]

plt.figure(figsize=(14, 8))

for feature_col in top_features_to_plot:
    ic_series = rolling_ic_dict[feature_col]
    
    if feature_col in highlight_features:
        plt.plot(
            ic_series.index,
            ic_series.values,
            linewidth=2,
            alpha=0.95,
            label=feature_col
        )
    else:
        plt.plot(
            ic_series.index,
            ic_series.values,
            linewidth=1,
            alpha=0.45,
            label=feature_col
        )

plt.axhline(0, linestyle="--", linewidth=1)
plt.title(f"Rolling IC Time Series: Top {top_plot_n} Features (Top {highlight_n} Highlighted)")
plt.xlabel("Date")
plt.ylabel("Spearman IC")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

In [ ]:
# Part 2 Step 5: First-Round Within-Group Candidate Screening
# Goal:
# First-round screening within comparable feature groups.
# This step does NOT finalize selected_feature_cols.
# It only creates group-level candidate features for later review.

# Main groups to screen:
# 1. lagged_return_features
# 2. trend_features
# 3. volatility_features
# 4. macro_change_features, split into VIX / DXY / TNX / OIL
# 5. macro_zscore_features, split into VIX / DXY / TNX / OIL
# 6. yield_curve_features
# 7. us_style_features, split into QQQ / IWM / DIA

top_k_per_group = 2

candidate_group_dict = {
    # Keep these as original groups
    "Lagged Return": lagged_return_features,
    "Trend": trend_features,
    "Volatility": volatility_features,

    # Split macro change features by asset family
    "VIX Return Lags": [
        col for col in macro_change_features
        if col.startswith("VIX_ret_")
    ],
    "DXY Return Lags": [
        col for col in macro_change_features
        if col.startswith("DXY_ret_")
    ],
    "TNX Change Lags": [
        col for col in macro_change_features
        if col.startswith("TNX_change_")
    ],
    "OIL Return Lags": [
        col for col in macro_change_features
        if col.startswith("OIL_ret_")
    ],

    # Split macro z-score features by asset family
    "VIX Z-score": [
        col for col in macro_zscore_features
        if col.startswith("VIX_z_")
    ],
    "DXY Z-score": [
        col for col in macro_zscore_features
        if col.startswith("DXY_z_")
    ],
    "TNX Z-score": [
        col for col in macro_zscore_features
        if col.startswith("TNX_z_")
    ],
    "OIL Z-score": [
        col for col in macro_zscore_features
        if col.startswith("OIL_z_")
    ],

    # Yield curve stays as one small group
    "Yield Curve": yield_curve_features,

    # Split U.S. style ETF features by ETF family
    "QQQ Return Lags": [
        col for col in us_style_features
        if col.startswith("QQQ_ret_")
    ],
    "IWM Return Lags": [
        col for col in us_style_features
        if col.startswith("IWM_ret_")
    ],
    "DIA Return Lags": [
        col for col in us_style_features
        if col.startswith("DIA_ret_")
    ],
}


def get_group_ic_result(group_features, rolling_ic_summary):
    """
    Return Rolling IC results for one feature group,
    sorted by IC stability and signal strength.
    """
    available_features = set(rolling_ic_summary["feature"])

    existing_features = [
        col for col in group_features
        if col in available_features
    ]

    group_result = (
        rolling_ic_summary[
            rolling_ic_summary["feature"].isin(existing_features)
        ]
        .sort_values(["abs_IC_IR", "abs_mean_IC"], ascending=False)
        .reset_index(drop=True)
    )

    return group_result


group_ic_results = {}
group_candidate_list = []

for group_name, group_features in candidate_group_dict.items():
    group_result = get_group_ic_result(
        group_features=group_features,
        rolling_ic_summary=rolling_ic_summary
    )

    group_ic_results[group_name] = group_result

    print(f"\n========== {group_name} ==========")
    print(f"Number of available features: {len(group_result)}")

    if len(group_result) == 0:
        print("No available Rolling IC result for this group.")
        continue

    display(
        group_result[
            [
                "feature",
                "mean_IC",
                "abs_mean_IC",
                "IC_std",
                "IC_IR",
                "abs_IC_IR",
                "positive_IC_share",
                "non_missing_windows"
            ]
        ]
    )

    # First-round candidates from this group
    temp_candidates = group_result.head(top_k_per_group).copy()
    temp_candidates.insert(0, "feature_group", group_name)
    group_candidate_list.append(temp_candidates)


if len(group_candidate_list) > 0:
    first_round_group_candidates = (
        pd.concat(group_candidate_list, axis=0)
        .reset_index(drop=True)
    )

    print("\n========== First-Round Group Candidates ==========")
    display(
        first_round_group_candidates[
            [
                "feature_group",
                "feature",
                "mean_IC",
                "abs_mean_IC",
                "IC_std",
                "IC_IR",
                "abs_IC_IR",
                "positive_IC_share",
                "non_missing_windows"
            ]
        ]
    )
else:
    first_round_group_candidates = pd.DataFrame()
    print("No first-round candidates were selected.")

In [ ]:
# First-Round Candidate Feature List
first_round_candidate_cols = (
    first_round_group_candidates["feature"]
    .drop_duplicates()
    .tolist()
)

print("Number of first-round candidate features:", len(first_round_candidate_cols))
print(first_round_candidate_cols)

In [ ]:
# Rolling IC Visualization: First-Round Candidates by Group
top_k_plot = 2

for group_name, group_result in group_ic_results.items():
    if len(group_result) == 0:
        continue

    features_to_plot = group_result.head(top_k_plot)["feature"].tolist()

    plt.figure(figsize=(12, 4.8))

    for feature_col in features_to_plot:
        if feature_col not in rolling_ic_dict:
            continue

        ic_series = rolling_ic_dict[feature_col]

        plt.plot(
            ic_series.index,
            ic_series.values,
            linewidth=1.5,
            alpha=0.85,
            label=feature_col
        )

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"Rolling IC: First-Round Candidates in {group_name}")
    plt.xlabel("Date")
    plt.ylabel("Spearman IC")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Part 2 Step 6: Build Second-Round Candidate Pool
# first_round_candidate_cols:
# candidates selected from within-group Rolling IC screening.
# Now add feature groups that were not included in within-group screening.

unscreened_group_dict = {
    "Momentum": momentum_features,
    "Bollinger": bollinger_features,
    "Intraday": intraday_features,
    "Volume": volume_features,
    "Calendar": calendar_features,
    "VIX Regime": regime_features
}

# VIX level is not the same as VIX_ret_* or VIX_z_*.
if "VIX" in train_cv_data.columns:
    unscreened_group_dict["VIX Level"] = ["VIX"]

available_ic_features = set(rolling_ic_summary["feature"])

unscreened_candidate_list = []

for group_name, group_features in unscreened_group_dict.items():
    existing_features = [
        col for col in group_features
        if col in available_ic_features
    ]
    
    if len(existing_features) == 0:
        continue
    
    temp = (
        rolling_ic_summary[
            rolling_ic_summary["feature"].isin(existing_features)
        ]
        .sort_values(["abs_IC_IR", "abs_mean_IC"], ascending=False)
        .copy()
    )
    
    temp.insert(0, "candidate_source", group_name)
    unscreened_candidate_list.append(temp)

if len(unscreened_candidate_list) > 0:
    unscreened_candidates = (
        pd.concat(unscreened_candidate_list, axis=0)
        .reset_index(drop=True)
    )
else:
    unscreened_candidates = pd.DataFrame()

print("Number of first-round within-group candidates:", len(first_round_candidate_cols))
print("Number of unscreened additional candidates:", len(unscreened_candidates))

display(
    unscreened_candidates[
        [
            "candidate_source",
            "feature",
            "mean_IC",
            "abs_mean_IC",
            "IC_std",
            "IC_IR",
            "abs_IC_IR",
            "positive_IC_share",
            "non_missing_windows"
        ]
    ]
)

In [ ]:
# Second-Round Candidate Pool
unscreened_candidate_cols = (
    unscreened_candidates["feature"]
    .drop_duplicates()
    .tolist()
    if len(unscreened_candidates) > 0
    else []
)

second_round_candidate_cols = list(
    dict.fromkeys(first_round_candidate_cols + unscreened_candidate_cols)
)

print("Number of second-round candidate features:", len(second_round_candidate_cols))
print(second_round_candidate_cols)

In [ ]:
# Part 2 Step 7: Label Second-Round Candidate Sources
# Candidates from within-group Rolling IC screening
within_group_source = (
    first_round_group_candidates[["feature_group", "feature"]]
    .rename(columns={"feature_group": "candidate_source"})
    .copy()
)

# Candidates from groups that were not screened in Step 5
unscreened_source = (
    unscreened_candidates[["candidate_source", "feature"]]
    .copy()
    if len(unscreened_candidates) > 0
    else pd.DataFrame(columns=["candidate_source", "feature"])
)

candidate_source_table = (
    pd.concat([within_group_source, unscreened_source], axis=0)
    .drop_duplicates(subset=["feature"])
    .reset_index(drop=True)
)

second_round_ic_summary = (
    rolling_ic_summary[
        rolling_ic_summary["feature"].isin(second_round_candidate_cols)
    ]
    .sort_values(["abs_IC_IR", "abs_mean_IC"], ascending=False)
    .reset_index(drop=True)
)

second_round_ic_summary_labeled = (
    second_round_ic_summary
    .merge(candidate_source_table, on="feature", how="left")
)

print("Second-round candidate pool size:", len(second_round_ic_summary_labeled))

display(
    second_round_ic_summary_labeled[
        [
            "candidate_source",
            "feature",
            "mean_IC",
            "abs_mean_IC",
            "IC_std",
            "IC_IR",
            "abs_IC_IR",
            "positive_IC_share",
            "non_missing_windows"
        ]
    ]
)

In [ ]:
# Part 2 Step 8: Category-Aware Global Screening
# Goal:
# Keep information diversity across feature categories.
# Rolling IC is used as a ranking/filtering tool, not as a simple global top-N rule.

min_abs_ic_ir = 0.10
min_abs_mean_ic = 0.015

category_keep_rules = {
    # Core technical families
    "Lagged Return": 2,
    "Trend": 2,
    "Volatility": 2,
    "Momentum": 2,
    "Bollinger": 1,

    # Macro change families
    "VIX Return Lags": 1,
    "DXY Return Lags": 1,
    "TNX Change Lags": 1,
    "OIL Return Lags": 1,

    # Macro state / z-score families
    "VIX Z-score": 1,
    "DXY Z-score": 1,
    "TNX Z-score": 1,
    "OIL Z-score": 1,

    # Yield curve
    "Yield Curve": 1,

    # Intraday / volume / calendar / regime
    "Intraday": 2,
    "Volume": 1,
    "Calendar": 1,
    "VIX Regime": 1,
    "VIX Level": 1,

    # U.S. style ETF families
    "QQQ Return Lags": 1,
    "IWM Return Lags": 1,
    "DIA Return Lags": 1
}

category_selected_list = []

for source, keep_n in category_keep_rules.items():
    temp = (
        second_round_ic_summary_labeled[
            second_round_ic_summary_labeled["candidate_source"] == source
        ]
        .copy()
        .sort_values(["abs_IC_IR", "abs_mean_IC"], ascending=False)
    )

    if len(temp) == 0:
        continue

    # Weak-signal filter.
    # A feature can pass if either IC stability or average IC strength is acceptable.
    temp_filtered = temp[
        (temp["abs_IC_IR"] >= min_abs_ic_ir) |
        (temp["abs_mean_IC"] >= min_abs_mean_ic)
    ].copy()

    if len(temp_filtered) == 0:
        continue

    category_selected_list.append(temp_filtered.head(keep_n))

if len(category_selected_list) > 0:
    category_aware_selected = (
        pd.concat(category_selected_list, axis=0)
        .drop_duplicates(subset=["feature"])
        .sort_values(["abs_IC_IR", "abs_mean_IC"], ascending=False)
        .reset_index(drop=True)
    )
else:
    category_aware_selected = pd.DataFrame()

print("Number of category-aware selected candidates:", len(category_aware_selected))

display(
    category_aware_selected[
        [
            "candidate_source",
            "feature",
            "mean_IC",
            "abs_mean_IC",
            "IC_std",
            "IC_IR",
            "abs_IC_IR",
            "positive_IC_share",
            "non_missing_windows"
        ]
    ]
)

category_aware_candidate_cols = category_aware_selected["feature"].tolist()

print("\ncategory_aware_candidate_cols:")
print(category_aware_candidate_cols)

In [ ]:
# Part 2 Step 9: Correlation Check among Category-Aware Candidates
candidate_corr = train_cv_data[category_aware_candidate_cols].corr().abs()

upper_triangle = candidate_corr.where(
    np.triu(np.ones(candidate_corr.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

high_corr_pairs.columns = ["feature_1", "feature_2", "abs_corr"]
high_corr_pairs = high_corr_pairs.sort_values("abs_corr", ascending=False)

print("Top high-correlation pairs among category-aware candidates:")
display(high_corr_pairs.head(30))

In [ ]:
# Visualization: High-Correlation Pairs among Candidates
top_corr_n = 20
plot_corr_df = high_corr_pairs.head(top_corr_n).copy()

plot_corr_df["pair"] = (
    plot_corr_df["feature_1"] + "  |  " + plot_corr_df["feature_2"]
)

plot_corr_df = plot_corr_df.sort_values("abs_corr", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(plot_corr_df["pair"], plot_corr_df["abs_corr"])
plt.xlabel("Absolute Correlation")
plt.ylabel("Feature Pair")
plt.title(f"Top {top_corr_n} High-Correlation Pairs")
plt.tight_layout()
plt.show()

In [ ]:
# Part 2 Step 10: Hierarchical Clustering for Correlation Cleanup
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

# Use category-aware candidates as the input pool for clustering cleanup
candidate_cols_for_clustering = [
    col for col in category_aware_candidate_cols
    if col in train_cv_data.columns
]

print("Number of candidates before clustering cleanup:", len(candidate_cols_for_clustering))

# 1. Compute absolute correlation matrix using train_cv_data only
candidate_corr = train_cv_data[candidate_cols_for_clustering].corr().abs()

# 2. Convert correlation to distance
# Higher abs correlation means smaller distance
distance_matrix = 1 - candidate_corr

# Avoid numerical issues on the diagonal
np.fill_diagonal(distance_matrix.values, 0)

# 3. Convert square distance matrix to condensed distance format
condensed_distance = squareform(distance_matrix.values, checks=False)

# 4. Hierarchical clustering
linkage_method = "average"

linkage_matrix = linkage(
    condensed_distance,
    method=linkage_method
)

# 5. Plot dendrogram
plt.figure(figsize=(15, 7))
dendrogram(
    linkage_matrix,
    labels=candidate_cols_for_clustering,
    leaf_rotation=90
)
plt.title("Hierarchical Clustering of Category-Aware Candidate Features")
plt.ylabel("Distance = 1 - abs(correlation)")
plt.tight_layout()
plt.show()

In [ ]:
# Cut Dendrogram into Redundancy Clusters
corr_threshold = 0.85
distance_threshold = 1 - corr_threshold

cluster_labels = fcluster(
    linkage_matrix,
    t=distance_threshold,
    criterion="distance"
)

feature_cluster_table = pd.DataFrame({
    "feature": candidate_cols_for_clustering,
    "cluster_id": cluster_labels
})

# Add IC metrics and candidate source
feature_cluster_table = (
    feature_cluster_table
    .merge(
        second_round_ic_summary_labeled[
            [
                "candidate_source",
                "feature",
                "mean_IC",
                "abs_mean_IC",
                "IC_std",
                "IC_IR",
                "abs_IC_IR",
                "positive_IC_share",
                "non_missing_windows"
            ]
        ],
        on="feature",
        how="left"
    )
    .sort_values(
        ["cluster_id", "abs_IC_IR", "abs_mean_IC"],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

print("Number of clusters:", feature_cluster_table["cluster_id"].nunique())

display(feature_cluster_table)

In [ ]:
# Cluster Summary
cluster_summary = (
    feature_cluster_table
    .groupby("cluster_id")
    .agg(
        n_features=("feature", "count"),
        features=("feature", lambda x: list(x)),
        candidate_sources=("candidate_source", lambda x: list(x))
    )
    .reset_index()
    .sort_values(["n_features", "cluster_id"], ascending=[False, True])
)

display(cluster_summary)

In [ ]:
# Inspect Features within Each Cluster
for cluster_id in sorted(feature_cluster_table["cluster_id"].unique()):
    temp = (
        feature_cluster_table[
            feature_cluster_table["cluster_id"] == cluster_id
        ]
        .sort_values(["abs_IC_IR", "abs_mean_IC"], ascending=False)
        .reset_index(drop=True)
    )

    print(f"\n========== Cluster {cluster_id} ==========")
    display(
        temp[
            [
                "candidate_source",
                "feature",
                "mean_IC",
                "abs_mean_IC",
                "IC_std",
                "IC_IR",
                "abs_IC_IR",
                "positive_IC_share",
                "non_missing_windows"
            ]
        ]
    )

In [ ]:
# Select Up to Two Representative Features from Large Clusters
cluster_selected_list = []

for cluster_id, temp in feature_cluster_table.groupby("cluster_id"):
    temp = temp.sort_values(
        ["abs_IC_IR", "abs_mean_IC"],
        ascending=False
    )

    cluster_size = len(temp)

    if cluster_size >= 3:
        keep_n = 2
    else:
        keep_n = 1

    cluster_selected_list.append(temp.head(keep_n))

cluster_representatives_2 = (
    pd.concat(cluster_selected_list, axis=0)
    .reset_index(drop=True)
)

selected_feature_cols_clustered_2 = (
    cluster_representatives_2
    .sort_values(["abs_IC_IR", "abs_mean_IC"], ascending=False)["feature"]
    .tolist()
)

print("Number of selected features with up to 2 per large cluster:",
      len(selected_feature_cols_clustered_2))
print(selected_feature_cols_clustered_2)

display(
    cluster_representatives_2[
        [
            "cluster_id",
            "candidate_source",
            "feature",
            "mean_IC",
            "abs_mean_IC",
            "IC_IR",
            "abs_IC_IR",
            "positive_IC_share",
            "non_missing_windows"
        ]
    ]
)

In [ ]:
# Correlation Check after Cluster-Based Cleanup
selected_feature_cols_clustered = selected_feature_cols_clustered_2.copy()

selected_corr_clustered = train_cv_data[selected_feature_cols_clustered].corr().abs()

upper_triangle = selected_corr_clustered.where(
    np.triu(np.ones(selected_corr_clustered.shape), k=1).astype(bool)
)

clustered_high_corr_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

clustered_high_corr_pairs.columns = ["feature_1", "feature_2", "abs_corr"]
clustered_high_corr_pairs = clustered_high_corr_pairs.sort_values(
    "abs_corr",
    ascending=False
)

print("Top high-correlation pairs after cluster-based cleanup:")
display(clustered_high_corr_pairs.head(20))

remaining_high_corr_pairs_clustered = clustered_high_corr_pairs[
    clustered_high_corr_pairs["abs_corr"] >= corr_threshold
]

print(f"\nRemaining pairs with abs_corr >= {corr_threshold}:",
      len(remaining_high_corr_pairs_clustered))

display(remaining_high_corr_pairs_clustered)

In [ ]:
# Correlation Heatmap after Cluster-Based Cleanup
plt.figure(figsize=(10, 8))
plt.imshow(selected_corr_clustered, aspect="auto")
plt.colorbar(label="Absolute Correlation")
plt.xticks(
    ticks=np.arange(len(selected_feature_cols_clustered)),
    labels=selected_feature_cols_clustered,
    rotation=90
)
plt.yticks(
    ticks=np.arange(len(selected_feature_cols_clustered)),
    labels=selected_feature_cols_clustered
)
plt.title("Absolute Correlation Matrix after Cluster-Based Cleanup")
plt.tight_layout()
plt.show()

In [ ]:
# Part 2 Step 10C: Final Manual Cleanup after Cluster-Based Screening
manual_drop_after_clustering = [
    "DIA_ret_5",     # highly correlated with ret_lag_5
    "SMA_gap_20"    # highly correlated with RSI_14 / SMA_gap_50
]

selected_feature_cols = [
    col for col in selected_feature_cols_clustered
    if col not in manual_drop_after_clustering
]

print("Number of selected features after final manual cleanup:", len(selected_feature_cols))
print(selected_feature_cols)

In [ ]:
# Final Correlation Check after Manual Cleanup
final_selected_corr = train_cv_data[selected_feature_cols].corr().abs()

upper_triangle = final_selected_corr.where(
    np.triu(np.ones(final_selected_corr.shape), k=1).astype(bool)
)

final_high_corr_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

final_high_corr_pairs.columns = ["feature_1", "feature_2", "abs_corr"]
final_high_corr_pairs = final_high_corr_pairs.sort_values(
    "abs_corr",
    ascending=False
)

print("Top high-correlation pairs after final manual cleanup:")
display(final_high_corr_pairs.head(20))

remaining_final_high_corr_pairs = final_high_corr_pairs[
    final_high_corr_pairs["abs_corr"] >= corr_threshold
]

print(f"\nRemaining pairs with abs_corr >= {corr_threshold}:",
      len(remaining_final_high_corr_pairs))

display(remaining_final_high_corr_pairs)

In [ ]:
# Part 2 Step 11: Final X/y Construction
X_train_cv = train_cv_data[selected_feature_cols].copy()
y_train_cv = train_cv_data["label"].copy()

X_test = test_data[selected_feature_cols].copy()
y_test = test_data["label"].copy()

print("========== Final X/y Shapes ==========")
print("X_train_cv shape:", X_train_cv.shape)
print("y_train_cv shape:", y_train_cv.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\n========== Train/CV Label Distribution ==========")
print(y_train_cv.value_counts().sort_index())
print(y_train_cv.value_counts(normalize=True).sort_index())

print("\n========== Test Label Distribution ==========")
print(y_test.value_counts().sort_index())
print(y_test.value_counts(normalize=True).sort_index())

In [ ]:
# Final Modeling Data Check
print("Missing values in X_train_cv:", X_train_cv.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

print("Infinite values in X_train_cv:", np.isinf(X_train_cv).sum().sum())
print("Infinite values in X_test:", np.isinf(X_test).sum().sum())

print("\nSelected feature count:", len(selected_feature_cols))
display(X_train_cv.describe().T)

## 6. Focused exploratory analysis

Inspect the training-period target, class balance, feature relationships, and regime stability without using the protected holdout for research decisions.

In [ ]:
# ============================================================
# Focused Train-Only EDA Setup
# ============================================================
label_to_signal = {-1: "Short", 0: "Flat", 1: "Long"}
signal_order = ["Short", "Flat", "Long"]

# Use only train_cv_data for EDA.
# Do not modify train_cv_data or test_data directly in the EDA section.
eda_data = train_cv_data.copy()
eda_data["signal"] = eda_data["label"].map(label_to_signal)

# Basic columns used in EDA
eda_target_cols = ["next_ret", "next_ret_vol_adj", "label", "signal"]
eda_feature_cols = selected_feature_cols.copy()

print("EDA data shape:", eda_data.shape)
print("Number of selected features:", len(eda_feature_cols))

display(eda_data[eda_target_cols].head())

In [ ]:
# Focused EDA 1: Train-Only Target and Label Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

eda_data["next_ret"].hist(bins=60, ax=axes[0])
axes[0].set_title("Train/CV Distribution of next_ret")
axes[0].set_xlabel("next_ret")
axes[0].set_ylabel("Frequency")

eda_data["next_ret_vol_adj"].hist(bins=60, ax=axes[1])
axes[1].set_title("Train/CV Distribution of next_ret_vol_adj")
axes[1].set_xlabel("next_ret_vol_adj")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

target_summary = eda_data[["next_ret", "next_ret_vol_adj"]].describe().T
target_summary["skew"] = eda_data[["next_ret", "next_ret_vol_adj"]].skew()
target_summary["kurtosis"] = eda_data[["next_ret", "next_ret_vol_adj"]].kurt()

display(target_summary)

In [ ]:
# Focused EDA 2: Selected Feature Correlation with Targets
target_corr_summary = []

for col in selected_feature_cols:
    pearson_next_ret = eda_data[[col, "next_ret"]].corr().iloc[0, 1]
    spearman_next_ret = eda_data[[col, "next_ret"]].corr(method="spearman").iloc[0, 1]

    pearson_vol_adj = eda_data[[col, "next_ret_vol_adj"]].corr().iloc[0, 1]
    spearman_vol_adj = eda_data[[col, "next_ret_vol_adj"]].corr(method="spearman").iloc[0, 1]

    target_corr_summary.append({
        "feature": col,
        "pearson_next_ret": pearson_next_ret,
        "spearman_next_ret": spearman_next_ret,
        "pearson_next_ret_vol_adj": pearson_vol_adj,
        "spearman_next_ret_vol_adj": spearman_vol_adj,
        "abs_spearman_vol_adj": abs(spearman_vol_adj)
    })

target_corr_summary = (
    pd.DataFrame(target_corr_summary)
    .sort_values("abs_spearman_vol_adj", ascending=False)
    .reset_index(drop=True)
)

display(target_corr_summary)

plot_df = target_corr_summary.head(15).sort_values("abs_spearman_vol_adj")

plt.figure(figsize=(9, 6))
plt.barh(plot_df["feature"], plot_df["spearman_next_ret_vol_adj"])
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Spearman Correlation with next_ret_vol_adj")
plt.title("Top Selected Features by Spearman Correlation with Vol-Adjusted Target")
plt.tight_layout()
plt.show()

In [ ]:
# Focused EDA 3: Selected Feature Summary by Signal Class
feature_by_class = (
    eda_data
    .groupby("signal")[selected_feature_cols]
    .agg(["mean", "median", "std"])
    .reindex(signal_order)
)

display(feature_by_class)

In [ ]:
# Focused EDA 4: Standardized Long-vs-Short Feature Differences
long_short_diff = []

for col in selected_feature_cols:
    long_mean = eda_data.loc[eda_data["signal"] == "Long", col].mean()
    short_mean = eda_data.loc[eda_data["signal"] == "Short", col].mean()
    train_std = eda_data[col].std()

    long_short_diff.append({
        "feature": col,
        "short_mean": short_mean,
        "long_mean": long_mean,
        "long_minus_short": long_mean - short_mean,
        "standardized_diff": (long_mean - short_mean) / train_std if train_std != 0 else np.nan
    })

long_short_diff = (
    pd.DataFrame(long_short_diff)
    .sort_values("standardized_diff", key=lambda x: x.abs(), ascending=False)
    .reset_index(drop=True)
)

display(long_short_diff)

plot_df = long_short_diff.head(15).sort_values("standardized_diff")

plt.figure(figsize=(9, 6))
plt.barh(plot_df["feature"], plot_df["standardized_diff"])
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Standardized Mean Difference: Long - Short")
plt.title("Selected Features: Long vs Short Difference")
plt.tight_layout()
plt.show()

In [ ]:
# Focused EDA 5: Representative Selected Features by Signal Class
eda_boxplot_features = [
    "SMA_gap_50",
    "RSI_14",
    "VIX",
    "Volatility_60",
    "high_low_range",
    "ret_lag_5",
    "DXY_ret_1",
    "TNX_change_5",
    "OIL_ret_1",
    "volume_ret_interaction"
]

eda_boxplot_features = [
    col for col in eda_boxplot_features
    if col in selected_feature_cols
]

n_features = len(eda_boxplot_features)

fig, axes = plt.subplots(n_features, 1, figsize=(10, 4 * n_features))

if n_features == 1:
    axes = [axes]

for ax, col in zip(axes, eda_boxplot_features):
    sns.boxplot(
        data=eda_data,
        x="signal",
        y=col,
        order=signal_order,
        ax=ax
    )
    ax.set_title(f"{col} by Signal Class (Train/CV Only)")
    ax.set_xlabel("Signal")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Focused EDA 6: Rolling Target and Class Regime
rolling_window = 60

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

eda_data["next_ret_vol_adj"].rolling(rolling_window).mean().plot(ax=axes[0])
axes[0].axhline(0, linestyle="--", linewidth=1)
axes[0].set_title(f"{rolling_window}-Day Rolling Mean of next_ret_vol_adj")
axes[0].set_ylabel("Rolling Mean")

eda_data["next_ret_vol_adj"].rolling(rolling_window).std().plot(ax=axes[1])
axes[1].set_title(f"{rolling_window}-Day Rolling Std of next_ret_vol_adj")
axes[1].set_ylabel("Rolling Std")

plt.tight_layout()
plt.show()

tmp = pd.get_dummies(eda_data["signal"])
rolling_class = tmp.rolling(rolling_window).mean()

rolling_class[signal_order].plot(figsize=(14, 5))
plt.title(f"{rolling_window}-Day Rolling Class Proportions (Train/CV Only)")
plt.ylabel("Proportion")
plt.tight_layout()
plt.show()

In [ ]:
# Focused EDA 8: Yearly Class Proportions
yearly_class = pd.crosstab(
    eda_data.index.year,
    eda_data["signal"],
    normalize="index"
)

yearly_class = yearly_class.reindex(columns=signal_order)

display(yearly_class)

yearly_class.plot(kind="bar", stacked=True, figsize=(12, 5))
plt.title("Yearly Class Proportions (Train/CV Only)")
plt.xlabel("Year")
plt.ylabel("Proportion")
plt.legend(title="Signal")
plt.tight_layout()
plt.show()

## 7. Modeling and walk-forward evaluation

Compare a majority-class benchmark with Logistic Regression, Random Forest, XGBoost, and LightGBM. Evaluate both classification quality and trading-oriented diagnostics at each validation fold.

In [ ]:
# ============================================================
# Modeling Step 0: Confirm Walk-Forward CV Setup
# ============================================================

walk_forward_summary = summarize_walk_forward_cv(
    data=train_cv_data,
    cv=walk_forward_cv
)

display(walk_forward_summary)

In [ ]:
# Modeling Step 1: Walk-Forward CV Evaluation Helper
cv_scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro"
}

def evaluate_model_walk_forward(model, X, y, cv, model_name):
    """
    Evaluate a model using the existing walk-forward CV object.
    This uses train_cv_data only and does not touch final test_data.
    """
    
    cv_results = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=cv_scoring,
        return_train_score=True,
        n_jobs=-1
    )
    
    summary_rows = []
    
    for metric in cv_scoring.keys():
        summary_rows.append({
            "model": model_name,
            "metric": metric,
            "train_mean": cv_results[f"train_{metric}"].mean(),
            "train_std": cv_results[f"train_{metric}"].std(),
            "cv_mean": cv_results[f"test_{metric}"].mean(),
            "cv_std": cv_results[f"test_{metric}"].std()
        })
    
    summary = pd.DataFrame(summary_rows)
    
    fold_results = pd.DataFrame({
        "fold": np.arange(1, cv.get_n_splits() + 1),
        "train_accuracy": cv_results["train_accuracy"],
        "cv_accuracy": cv_results["test_accuracy"],
        "train_balanced_accuracy": cv_results["train_balanced_accuracy"],
        "cv_balanced_accuracy": cv_results["test_balanced_accuracy"],
        "train_macro_f1": cv_results["train_macro_f1"],
        "cv_macro_f1": cv_results["test_macro_f1"]
    })
    
    fold_results.insert(0, "model", model_name)
    
    return summary, fold_results, cv_results

In [ ]:
# Classification Evaluation Helper
def evaluate_classification(y_true, y_pred, model_name="model"):
    """
    Classification evaluation for Long / Flat / Short signal prediction.
    Labels: -1 = Short, 0 = Flat, 1 = Long
    """
    
    labels = [-1, 0, 1]
    label_names = ["Short", "Flat", "Long"]
    
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    
    print("=" * 70)
    print(f"Classification Evaluation: {model_name}")
    print("=" * 70)
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    
    print("\nPrediction Distribution:")
    pred_dist = pd.Series(y_pred).value_counts(normalize=True).sort_index()
    display(pred_dist)
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{name}" for name in label_names],
        columns=[f"Pred_{name}" for name in label_names]
    )
    display(cm_df)
    
    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=label_names,
        digits=4,
        zero_division=0
    ))
    
    summary = {
        "model": model_name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc
    }
    
    return summary, cm_df

In [ ]:
# Trading-Oriented Evaluation Helper
def evaluate_trading_performance(test_data, y_true, y_pred, model_name="model"):
    """
    Trading-oriented evaluation.
    Position rule:
        1  = Long SPY
        0  = Flat
        -1 = Short SPY
        
    Strategy return:
        position[t] * next_ret[t]
    """
    
    eval_df = test_data.copy()
    eval_df["true_label"] = y_true.values if hasattr(y_true, "values") else y_true
    eval_df["pred_label"] = y_pred
    eval_df["position"] = eval_df["pred_label"]
    eval_df["strategy_ret"] = eval_df["position"] * eval_df["next_ret"]
    
    # Only active trading days
    eval_df["is_active_trade"] = eval_df["position"] != 0
    
    daily_mean = eval_df["strategy_ret"].mean()
    daily_vol = eval_df["strategy_ret"].std()
    
    ann_return = daily_mean * 252
    ann_vol = daily_vol * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    
    cumulative_ret = eval_df["strategy_ret"].cumsum()
    running_max = cumulative_ret.cummax()
    drawdown = cumulative_ret - running_max
    max_drawdown = drawdown.min()
    
    # Hit rate: among active trades, did the position direction match realized next_ret?
    active_df = eval_df[eval_df["is_active_trade"]].copy()
    
    if len(active_df) > 0:
        hit_rate = (
            np.sign(active_df["position"]) == np.sign(active_df["next_ret"])
        ).mean()
    else:
        hit_rate = np.nan
    
    # False Long: predicted Long but true label is Short
    long_pred_count = (eval_df["pred_label"] == 1).sum()
    false_long_count = (
        (eval_df["pred_label"] == 1) &
        (eval_df["true_label"] == -1)
    ).sum()
    
    false_long_rate = (
        false_long_count / long_pred_count
        if long_pred_count > 0
        else np.nan
    )
    
    signal_distribution = (
        eval_df["pred_label"]
        .value_counts(normalize=True)
        .sort_index()
        .rename(index={-1: "Short", 0: "Flat", 1: "Long"})
    )
    
    trading_summary = pd.DataFrame({
        "model": [model_name],
        "mean_daily_return": [daily_mean],
        "annualized_return": [ann_return],
        "annualized_volatility": [ann_vol],
        "sharpe": [sharpe],
        "max_drawdown": [max_drawdown],
        "hit_rate_active_trades": [hit_rate],
        "false_long_count": [false_long_count],
        "long_pred_count": [long_pred_count],
        "false_long_rate": [false_long_rate],
        "active_trade_rate": [eval_df["is_active_trade"].mean()]
    })
    
    print("=" * 70)
    print(f"Trading-Oriented Evaluation: {model_name}")
    print("=" * 70)
    display(trading_summary)
    
    print("\nSignal Distribution:")
    display(signal_distribution)
    
    return trading_summary, eval_df

In [ ]:
# Modeling Step 2: Majority-Class Baseline
majority_pipe = Pipeline([
    ("model", DummyClassifier(strategy="most_frequent"))
])

majority_summary, majority_folds, majority_cv_results = evaluate_model_walk_forward(
    model=majority_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Majority Class Baseline"
)

display(majority_summary)
display(majority_folds)

In [ ]:
# ============================================================
# Modeling Step 3: Baseline Logistic Regression
# ============================================================
# Fixed baseline model:
# - Uses StandardScaler because Logistic Regression is scale-sensitive.
# - No hyperparameter tuning yet.
# - This is the clean interpretable benchmark after majority-class baseline.

logistic_baseline_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        penalty="l2",
        C=1.0,
        class_weight=None,
        random_state=42
    ))
])

logistic_baseline_summary, logistic_baseline_folds, logistic_baseline_cv_results = evaluate_model_walk_forward(
    model=logistic_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Baseline Logistic Regression"
)

display(logistic_baseline_summary)
display(logistic_baseline_folds)

In [ ]:
# ============================================================
# Modeling Step 4: Tuned Logistic Regression
# ============================================================
# FULL_TUNING=True searches only within train_cv_data.
# Fast mode reuses parameters selected by that original search.

logistic_tune_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        penalty="l2",
        random_state=42
    ))
])

logistic_param_grid = {
    "model__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
    "model__class_weight": [None, "balanced"]
}

if FULL_TUNING:
    logistic_grid = GridSearchCV(
        estimator=logistic_tune_pipe,
        param_grid=logistic_param_grid,
        scoring=cv_scoring,
        refit="balanced_accuracy",
        cv=walk_forward_cv,
        return_train_score=True,
        n_jobs=-1
    )
    logistic_grid.fit(X_train_cv, y_train_cv)
    tuned_logistic_pipe = logistic_grid.best_estimator_
    print("Best Logistic Regression Parameters:")
    print(logistic_grid.best_params_)
    print()
    print("Best CV Balanced Accuracy:")
    print(logistic_grid.best_score_)
else:
    logistic_grid = None
    tuned_logistic_pipe = logistic_tune_pipe.set_params(
        **RECORDED_BEST_PARAMS["logistic"]
    )
    print("Fast mode: using recorded Logistic Regression parameters:")
    print(RECORDED_BEST_PARAMS["logistic"])

tuned_logistic_summary, tuned_logistic_folds, tuned_logistic_cv_results = evaluate_model_walk_forward(
    model=tuned_logistic_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Tuned Logistic Regression"
)

display(tuned_logistic_summary)
display(tuned_logistic_folds)

In [ ]:
# ============================================================
# Modeling Step 5: Compare Logistic Models Using Walk-Forward CV
# ============================================================

model_cv_summary = pd.concat(
    [
        majority_summary,
        logistic_baseline_summary,
        tuned_logistic_summary
    ],
    axis=0
).reset_index(drop=True)

display(model_cv_summary)

model_cv_pivot = model_cv_summary.pivot(
    index="model",
    columns="metric",
    values="cv_mean"
).sort_values("balanced_accuracy", ascending=False)

display(model_cv_pivot)

fold_level_comparison = pd.concat(
    [
        majority_folds,
        logistic_baseline_folds,
        tuned_logistic_folds
    ],
    axis=0
).reset_index(drop=True)

display(fold_level_comparison)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=model_cv_summary[model_cv_summary["metric"].isin(["balanced_accuracy", "macro_f1"])],
    x="model",
    y="cv_mean",
    hue="metric"
)
plt.xticks(rotation=30, ha="right")
plt.title("Walk-Forward CV Performance: Baseline Models")
plt.ylabel("CV Mean Score")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=fold_level_comparison,
    x="fold",
    y="cv_balanced_accuracy",
    hue="model",
    marker="o"
)
plt.title("Fold-Level CV Balanced Accuracy")
plt.ylabel("CV Balanced Accuracy")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=fold_level_comparison,
    x="fold",
    y="cv_macro_f1",
    hue="model",
    marker="o"
)
plt.title("Fold-Level CV Macro F1")
plt.ylabel("CV Macro F1")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Modeling Step 6: Walk-Forward CV Trading Diagnostic
# ============================================================
# This evaluates trading behavior on validation folds inside train_cv_data only.
# It is not final test performance.

from sklearn.base import clone

def evaluate_walk_forward_trading(model, X, y, data, cv, model_name):
    """
    Walk-forward validation-fold trading diagnostic.
    Uses only train_cv_data validation folds.
    """
    
    fold_rows = []
    all_eval_parts = []
    
    for fold_id, (train_idx, val_idx) in enumerate(cv.split(X), start=1):
        model_fold = clone(model)
        
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_val_fold = y.iloc[val_idx]
        
        val_data = data.iloc[val_idx].copy()
        
        model_fold.fit(X_train_fold, y_train_fold)
        y_val_pred = model_fold.predict(X_val_fold)
        
        eval_df = val_data.copy()
        eval_df["fold"] = fold_id
        eval_df["true_label"] = y_val_fold.values
        eval_df["pred_label"] = y_val_pred
        eval_df["position"] = eval_df["pred_label"]
        eval_df["strategy_ret"] = eval_df["position"] * eval_df["next_ret"]
        eval_df["is_active_trade"] = eval_df["position"] != 0
        
        daily_mean = eval_df["strategy_ret"].mean()
        daily_vol = eval_df["strategy_ret"].std()
        
        ann_return = daily_mean * 252
        ann_vol = daily_vol * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
        
        cumulative_ret = eval_df["strategy_ret"].cumsum()
        running_max = cumulative_ret.cummax()
        drawdown = cumulative_ret - running_max
        max_drawdown = drawdown.min()
        
        active_df = eval_df[eval_df["is_active_trade"]].copy()
        hit_rate = (
            np.sign(active_df["position"]) == np.sign(active_df["next_ret"])
        ).mean() if len(active_df) > 0 else np.nan
        
        long_pred_count = (eval_df["pred_label"] == 1).sum()
        false_long_count = (
            (eval_df["pred_label"] == 1) &
            (eval_df["true_label"] == -1)
        ).sum()
        
        false_long_rate = (
            false_long_count / long_pred_count
            if long_pred_count > 0
            else np.nan
        )
        
        fold_rows.append({
            "model": model_name,
            "fold": fold_id,
            "mean_daily_return": daily_mean,
            "annualized_return": ann_return,
            "annualized_volatility": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_drawdown,
            "hit_rate_active_trades": hit_rate,
            "false_long_count": false_long_count,
            "long_pred_count": long_pred_count,
            "false_long_rate": false_long_rate,
            "active_trade_rate": eval_df["is_active_trade"].mean(),
            "short_pred_share": (eval_df["pred_label"] == -1).mean(),
            "flat_pred_share": (eval_df["pred_label"] == 0).mean(),
            "long_pred_share": (eval_df["pred_label"] == 1).mean()
        })
        
        all_eval_parts.append(eval_df)
    
    fold_trading_summary = pd.DataFrame(fold_rows)
    all_eval_df = pd.concat(all_eval_parts, axis=0)
    
    overall_summary = fold_trading_summary.drop(columns=["fold"]).groupby("model").mean().reset_index()
    
    return overall_summary, fold_trading_summary, all_eval_df


majority_trading_summary, majority_trading_folds, majority_cv_trade_df = evaluate_walk_forward_trading(
    model=majority_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Majority Class Baseline"
)

logistic_baseline_trading_summary, logistic_baseline_trading_folds, logistic_baseline_cv_trade_df = evaluate_walk_forward_trading(
    model=logistic_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Baseline Logistic Regression"
)

tuned_logistic_trading_summary, tuned_logistic_trading_folds, tuned_logistic_cv_trade_df = evaluate_walk_forward_trading(
    model=tuned_logistic_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Tuned Logistic Regression"
)

logistic_cv_trading_comparison = pd.concat(
    [
        majority_trading_summary,
        logistic_baseline_trading_summary,
        tuned_logistic_trading_summary
    ],
    axis=0
).reset_index(drop=True)

display(logistic_cv_trading_comparison)

logistic_cv_trading_folds = pd.concat(
    [
        majority_trading_folds,
        logistic_baseline_trading_folds,
        tuned_logistic_trading_folds
    ],
    axis=0
).reset_index(drop=True)

display(logistic_cv_trading_folds)

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=logistic_cv_trading_folds,
    x="fold",
    y="sharpe",
    hue="model",
    marker="o"
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Walk-Forward CV Trading Sharpe by Fold")
plt.ylabel("Sharpe")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=logistic_cv_trading_folds,
    x="fold",
    y="false_long_rate",
    hue="model",
    marker="o"
)
plt.title("Walk-Forward CV False Long Rate by Fold")
plt.ylabel("False Long Rate")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 8: Random Forest Baseline
# Tree-based models do not require StandardScaler.
# This is a fixed baseline Random Forest, separate from the tuned version.
rf_baseline_pipe = Pipeline([
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

rf_baseline_summary, rf_baseline_folds, rf_baseline_cv_results = evaluate_model_walk_forward(
    model=rf_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Baseline Random Forest"
)

display(rf_baseline_summary)
display(rf_baseline_folds)

In [ ]:
# Modeling Step 9: Tuned Random Forest
# FULL_TUNING=True searches only within train_cv_data.
# Fast mode reuses parameters selected by that original search.
rf_tune_pipe = Pipeline([
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        bootstrap=True
    ))
])

rf_param_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [2, 3, 4, 5, None],
    "model__min_samples_split": [10, 20, 40],
    "model__min_samples_leaf": [5, 10, 20],
    "model__max_features": ["sqrt", 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"]
}

if FULL_TUNING:
    rf_grid = GridSearchCV(
        estimator=rf_tune_pipe,
        param_grid=rf_param_grid,
        scoring=cv_scoring,
        refit="balanced_accuracy",
        cv=walk_forward_cv,
        return_train_score=True,
        n_jobs=-1,
        verbose=1
    )
    rf_grid.fit(X_train_cv, y_train_cv)
    tuned_rf_pipe = rf_grid.best_estimator_
    print("Best Random Forest Parameters:")
    print(rf_grid.best_params_)
    print()
    print("Best CV Balanced Accuracy:")
    print(rf_grid.best_score_)
else:
    rf_grid = None
    tuned_rf_pipe = rf_tune_pipe.set_params(
        **RECORDED_BEST_PARAMS["random_forest"]
    )
    print("Fast mode: using recorded Random Forest parameters:")
    print(RECORDED_BEST_PARAMS["random_forest"])

tuned_rf_summary, tuned_rf_folds, tuned_rf_cv_results = evaluate_model_walk_forward(
    model=tuned_rf_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Tuned Random Forest"
)

display(tuned_rf_summary)
display(tuned_rf_folds)

In [ ]:
# Modeling Step 10: Random Forest Tuning Results Review
if FULL_TUNING:
    rf_grid_results = pd.DataFrame(rf_grid.cv_results_)

    rf_grid_display_cols = [
        "mean_train_balanced_accuracy",
        "std_train_balanced_accuracy",
        "mean_test_balanced_accuracy",
        "std_test_balanced_accuracy",
        "mean_train_macro_f1",
        "std_train_macro_f1",
        "mean_test_macro_f1",
        "std_test_macro_f1",
        "mean_test_accuracy",
        "params"
    ]

    rf_grid_review = (
        rf_grid_results[rf_grid_display_cols]
        .sort_values("mean_test_balanced_accuracy", ascending=False)
        .reset_index(drop=True)
    )
    display(rf_grid_review.head(15))
else:
    print("Fast mode: exhaustive Random Forest tuning table skipped.")
    display(pd.Series(
        RECORDED_BEST_PARAMS["random_forest"],
        name="recorded_value"
    ).to_frame())

In [ ]:
# Modeling Step 11: Compare Logistic and Random Forest CV Results
model_cv_summary = pd.concat(
    [
        majority_summary,
        logistic_baseline_summary,
        tuned_logistic_summary,
        rf_baseline_summary,
        tuned_rf_summary
    ],
    axis=0
).reset_index(drop=True)

display(model_cv_summary)

model_cv_pivot = model_cv_summary.pivot(
    index="model",
    columns="metric",
    values="cv_mean"
).sort_values("balanced_accuracy", ascending=False)

display(model_cv_pivot)

fold_level_comparison = pd.concat(
    [
        majority_folds,
        logistic_baseline_folds,
        tuned_logistic_folds,
        rf_baseline_folds,
        tuned_rf_folds
    ],
    axis=0
).reset_index(drop=True)

display(fold_level_comparison)

plt.figure(figsize=(11, 5))
sns.barplot(
    data=model_cv_summary[model_cv_summary["metric"].isin(["balanced_accuracy", "macro_f1"])],
    x="model",
    y="cv_mean",
    hue="metric"
)
plt.xticks(rotation=30, ha="right")
plt.title("Walk-Forward CV Performance: Logistic vs Random Forest")
plt.ylabel("CV Mean Score")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=fold_level_comparison,
    x="fold",
    y="cv_balanced_accuracy",
    hue="model",
    marker="o"
)
plt.title("Fold-Level CV Balanced Accuracy")
plt.ylabel("CV Balanced Accuracy")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=fold_level_comparison,
    x="fold",
    y="cv_macro_f1",
    hue="model",
    marker="o"
)
plt.title("Fold-Level CV Macro F1")
plt.ylabel("CV Macro F1")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 12: Random Forest Walk-Forward Trading Diagnostic
# This evaluates validation-fold trading behavior inside train_cv_data only.
# It is not final test performance.
rf_baseline_trading_summary, rf_baseline_trading_folds, rf_baseline_cv_trade_df = evaluate_walk_forward_trading(
    model=rf_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Baseline Random Forest"
)

tuned_rf_trading_summary, tuned_rf_trading_folds, tuned_rf_cv_trade_df = evaluate_walk_forward_trading(
    model=tuned_rf_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Tuned Random Forest"
)

model_cv_trading_comparison = pd.concat(
    [
        majority_trading_summary,
        logistic_baseline_trading_summary,
        tuned_logistic_trading_summary,
        rf_baseline_trading_summary,
        tuned_rf_trading_summary
    ],
    axis=0
).reset_index(drop=True)

display(model_cv_trading_comparison)

model_cv_trading_folds = pd.concat(
    [
        majority_trading_folds,
        logistic_baseline_trading_folds,
        tuned_logistic_trading_folds,
        rf_baseline_trading_folds,
        tuned_rf_trading_folds
    ],
    axis=0
).reset_index(drop=True)

display(model_cv_trading_folds)

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=model_cv_trading_folds,
    x="fold",
    y="sharpe",
    hue="model",
    marker="o"
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Walk-Forward CV Trading Sharpe by Fold")
plt.ylabel("Sharpe")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=model_cv_trading_folds,
    x="fold",
    y="false_long_rate",
    hue="model",
    marker="o"
)
plt.title("Walk-Forward CV False Long Rate by Fold")
plt.ylabel("False Long Rate")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=model_cv_trading_folds,
    x="fold",
    y="active_trade_rate",
    hue="model",
    marker="o"
)
plt.title("Walk-Forward CV Active Trade Rate by Fold")
plt.ylabel("Active Trade Rate")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 13: Random Forest Feature Importance
# This is fitted on train_cv_data only for interpretation.
# It is not final test evaluation.
tuned_rf_pipe.fit(X_train_cv, y_train_cv)

rf_feature_importance = pd.DataFrame({
    "feature": selected_feature_cols,
    "importance": tuned_rf_pipe.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

display(rf_feature_importance)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=rf_feature_importance.head(15).sort_values("importance"),
    x="importance",
    y="feature"
)
plt.title("Tuned Random Forest: Top 15 Feature Importances")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 14: XGBoost / LightGBM Label Wrapper
# XGBoost / LightGBM internally use encoded labels:
# -1 -> 0
#  0 -> 1
#  1 -> 2
#
# The wrapper converts predictions back to the original labels:
# 0 -> -1
# 1 ->  0
# 2 ->  1
#
# This keeps all existing evaluation functions unchanged.
label_to_encoded = {-1: 0, 0: 1, 1: 2}
encoded_to_label = {0: -1, 1: 0, 2: 1}


class XGBSignalClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=0.0,
        reg_lambda=5.0,
        reg_alpha=0.0,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,
        verbosity=0
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.min_child_weight = min_child_weight
        self.gamma = gamma
        self.reg_lambda = reg_lambda
        self.reg_alpha = reg_alpha
        self.objective = objective
        self.eval_metric = eval_metric
        self.tree_method = tree_method
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.verbosity = verbosity

    def fit(self, X, y):
        y_encoded = pd.Series(y).map(label_to_encoded).values

        self.model_ = XGBClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree,
            min_child_weight=self.min_child_weight,
            gamma=self.gamma,
            reg_lambda=self.reg_lambda,
            reg_alpha=self.reg_alpha,
            objective=self.objective,
            eval_metric=self.eval_metric,
            tree_method=self.tree_method,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
            verbosity=self.verbosity
        )

        self.model_.fit(X, y_encoded)
        self.classes_ = np.array([-1, 0, 1])
        return self

    def predict(self, X):
        pred_encoded = self.model_.predict(X)
        return pd.Series(pred_encoded).map(encoded_to_label).values

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

In [ ]:
# Modeling Step 15: XGBoost Baseline
# Tree-based boosting model. No StandardScaler is needed.
# This baseline is intentionally conservative to reduce overfitting risk.
xgb_baseline_pipe = Pipeline([
    ("model", XGBSignalClassifier(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        gamma=0.0,
        reg_lambda=10.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=1,
        verbosity=0
    ))
])

xgb_baseline_summary, xgb_baseline_folds, xgb_baseline_cv_results = evaluate_model_walk_forward(
    model=xgb_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Baseline XGBoost"
)

display(xgb_baseline_summary)
display(xgb_baseline_folds)

In [ ]:
# Modeling Step 16: Tuned XGBoost
# FULL_TUNING=True searches only within train_cv_data.
# Fast mode reuses parameters selected by that original search.
xgb_tune_pipe = Pipeline([
    ("model", XGBSignalClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,
        verbosity=0
    ))
])

xgb_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [1, 2, 3],
    "model__learning_rate": [0.01, 0.03, 0.05],
    "model__subsample": [0.7, 0.9],
    "model__colsample_bytree": [0.7, 0.9],
    "model__min_child_weight": [5, 10, 20],
    "model__reg_lambda": [5.0, 10.0, 20.0],
    "model__reg_alpha": [0.0, 0.1]
}

if FULL_TUNING:
    xgb_grid = GridSearchCV(
        estimator=xgb_tune_pipe,
        param_grid=xgb_param_grid,
        scoring=cv_scoring,
        refit="balanced_accuracy",
        cv=walk_forward_cv,
        return_train_score=True,
        n_jobs=-1,
        verbose=1
    )
    xgb_grid.fit(X_train_cv, y_train_cv)
    tuned_xgb_pipe = xgb_grid.best_estimator_
    print("Best XGBoost Parameters:")
    print(xgb_grid.best_params_)
    print()
    print("Best CV Balanced Accuracy:")
    print(xgb_grid.best_score_)
else:
    xgb_grid = None
    tuned_xgb_pipe = xgb_tune_pipe.set_params(
        **RECORDED_BEST_PARAMS["xgboost"]
    )
    print("Fast mode: using recorded XGBoost parameters:")
    print(RECORDED_BEST_PARAMS["xgboost"])

tuned_xgb_summary, tuned_xgb_folds, tuned_xgb_cv_results = evaluate_model_walk_forward(
    model=tuned_xgb_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Tuned XGBoost"
)

display(tuned_xgb_summary)
display(tuned_xgb_folds)

In [ ]:
# Modeling Step 17: XGBoost Tuning Results Review
if FULL_TUNING:
    xgb_grid_results = pd.DataFrame(xgb_grid.cv_results_)

    xgb_grid_display_cols = [
        "mean_train_balanced_accuracy",
        "std_train_balanced_accuracy",
        "mean_test_balanced_accuracy",
        "std_test_balanced_accuracy",
        "mean_train_macro_f1",
        "std_train_macro_f1",
        "mean_test_macro_f1",
        "std_test_macro_f1",
        "mean_test_accuracy",
        "params"
    ]

    xgb_grid_review = (
        xgb_grid_results[xgb_grid_display_cols]
        .sort_values("mean_test_balanced_accuracy", ascending=False)
        .reset_index(drop=True)
    )
    display(xgb_grid_review.head(15))
else:
    print("Fast mode: exhaustive XGBoost tuning table skipped.")
    display(pd.Series(
        RECORDED_BEST_PARAMS["xgboost"],
        name="recorded_value"
    ).to_frame())

In [ ]:
# Modeling Step 18: XGBoost Walk-Forward Trading Diagnostic
# This evaluates validation-fold trading behavior inside train_cv_data only.
# It is not final test performance.
xgb_baseline_trading_summary, xgb_baseline_trading_folds, xgb_baseline_cv_trade_df = evaluate_walk_forward_trading(
    model=xgb_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Baseline XGBoost"
)

tuned_xgb_trading_summary, tuned_xgb_trading_folds, tuned_xgb_cv_trade_df = evaluate_walk_forward_trading(
    model=tuned_xgb_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Tuned XGBoost"
)

display(xgb_baseline_trading_summary)
display(tuned_xgb_trading_summary)

display(xgb_baseline_trading_folds)
display(tuned_xgb_trading_folds)

In [ ]:
# Modeling Step 19: XGBoost Feature Importance
# This is fitted on train_cv_data only for interpretation.
# It is not final test evaluation.
tuned_xgb_pipe.fit(X_train_cv, y_train_cv)

xgb_feature_importance = pd.DataFrame({
    "feature": selected_feature_cols,
    "importance": tuned_xgb_pipe.named_steps["model"].model_.feature_importances_
}).sort_values("importance", ascending=False)

display(xgb_feature_importance)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=xgb_feature_importance.head(15).sort_values("importance"),
    x="importance",
    y="feature"
)
plt.title("Tuned XGBoost: Top 15 Feature Importances")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 20: LightGBM Label Wrapper
# LightGBM internally uses encoded labels:
# -1 -> 0
#  0 -> 1
#  1 -> 2
#
# Predictions are converted back to original labels:
# 0 -> -1
# 1 ->  0
# 2 ->  1
#
# This keeps existing evaluation functions unchanged.
class LGBMSignalClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        num_leaves=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=30,
        reg_lambda=10.0,
        reg_alpha=0.0,
        class_weight=None,
        objective="multiclass",
        random_state=42,
        n_jobs=1,
        verbose=-1
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.min_child_samples = min_child_samples
        self.reg_lambda = reg_lambda
        self.reg_alpha = reg_alpha
        self.class_weight = class_weight
        self.objective = objective
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.verbose = verbose

    def fit(self, X, y):
        y_encoded = pd.Series(y).map(label_to_encoded).values

        self.model_ = LGBMClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            num_leaves=self.num_leaves,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree,
            min_child_samples=self.min_child_samples,
            reg_lambda=self.reg_lambda,
            reg_alpha=self.reg_alpha,
            class_weight=self.class_weight,
            objective=self.objective,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
            verbose=self.verbose
        )

        self.model_.fit(X, y_encoded)
        self.classes_ = np.array([-1, 0, 1])
        return self

    def predict(self, X):
        pred_encoded = self.model_.predict(X)
        return pd.Series(pred_encoded).map(encoded_to_label).values

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

In [ ]:
# Modeling Step 21: LightGBM Baseline
# Tree-based boosting model. No StandardScaler is needed.
# This baseline is intentionally conservative due to short sample window.
lgbm_baseline_pipe = Pipeline([
    ("model", LGBMSignalClassifier(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        num_leaves=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=30,
        reg_lambda=10.0,
        reg_alpha=0.0,
        class_weight=None,
        random_state=42,
        n_jobs=1,
        verbose=-1
    ))
])

lgbm_baseline_summary, lgbm_baseline_folds, lgbm_baseline_cv_results = evaluate_model_walk_forward(
    model=lgbm_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Baseline LightGBM"
)

display(lgbm_baseline_summary)
display(lgbm_baseline_folds)

In [ ]:
# Modeling Step 22: Tuned LightGBM
# FULL_TUNING=True searches only within train_cv_data.
# Fast mode reuses parameters selected by that original search.
lgbm_tune_pipe = Pipeline([
    ("model", LGBMSignalClassifier(
        objective="multiclass",
        random_state=42,
        n_jobs=1,
        verbose=-1
    ))
])

lgbm_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [1, 2, 3],
    "model__learning_rate": [0.01, 0.03, 0.05],
    "model__num_leaves": [3, 4, 7],
    "model__subsample": [0.7, 0.9],
    "model__colsample_bytree": [0.7, 0.9],
    "model__min_child_samples": [20, 30, 50],
    "model__reg_lambda": [5.0, 10.0, 20.0],
    "model__reg_alpha": [0.0, 0.1],
    "model__class_weight": [None, "balanced"]
}

if FULL_TUNING:
    lgbm_grid = GridSearchCV(
        estimator=lgbm_tune_pipe,
        param_grid=lgbm_param_grid,
        scoring=cv_scoring,
        refit="balanced_accuracy",
        cv=walk_forward_cv,
        return_train_score=True,
        n_jobs=-1,
        verbose=1
    )
    lgbm_grid.fit(X_train_cv, y_train_cv)
    tuned_lgbm_pipe = lgbm_grid.best_estimator_
    print("Best LightGBM Parameters:")
    print(lgbm_grid.best_params_)
    print()
    print("Best CV Balanced Accuracy:")
    print(lgbm_grid.best_score_)
else:
    lgbm_grid = None
    tuned_lgbm_pipe = lgbm_tune_pipe.set_params(
        **RECORDED_BEST_PARAMS["lightgbm"]
    )
    print("Fast mode: using recorded LightGBM parameters:")
    print(RECORDED_BEST_PARAMS["lightgbm"])

tuned_lgbm_summary, tuned_lgbm_folds, tuned_lgbm_cv_results = evaluate_model_walk_forward(
    model=tuned_lgbm_pipe,
    X=X_train_cv,
    y=y_train_cv,
    cv=walk_forward_cv,
    model_name="Tuned LightGBM"
)

display(tuned_lgbm_summary)
display(tuned_lgbm_folds)

In [ ]:
# Modeling Step 23: LightGBM Tuning Results Review
if FULL_TUNING:
    lgbm_grid_results = pd.DataFrame(lgbm_grid.cv_results_)

    lgbm_grid_display_cols = [
        "mean_train_balanced_accuracy",
        "std_train_balanced_accuracy",
        "mean_test_balanced_accuracy",
        "std_test_balanced_accuracy",
        "mean_train_macro_f1",
        "std_train_macro_f1",
        "mean_test_macro_f1",
        "std_test_macro_f1",
        "mean_test_accuracy",
        "params"
    ]

    lgbm_grid_review = (
        lgbm_grid_results[lgbm_grid_display_cols]
        .sort_values("mean_test_balanced_accuracy", ascending=False)
        .reset_index(drop=True)
    )
    display(lgbm_grid_review.head(15))
else:
    print("Fast mode: exhaustive LightGBM tuning table skipped.")
    display(pd.Series(
        RECORDED_BEST_PARAMS["lightgbm"],
        name="recorded_value"
    ).to_frame())

In [ ]:
# Modeling Step 24: LightGBM Walk-Forward Trading Diagnostic
# This evaluates validation-fold trading behavior inside train_cv_data only.
# It is not final test performance.
lgbm_baseline_trading_summary, lgbm_baseline_trading_folds, lgbm_baseline_cv_trade_df = evaluate_walk_forward_trading(
    model=lgbm_baseline_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Baseline LightGBM"
)

tuned_lgbm_trading_summary, tuned_lgbm_trading_folds, tuned_lgbm_cv_trade_df = evaluate_walk_forward_trading(
    model=tuned_lgbm_pipe,
    X=X_train_cv,
    y=y_train_cv,
    data=train_cv_data,
    cv=walk_forward_cv,
    model_name="Tuned LightGBM"
)

display(lgbm_baseline_trading_summary)
display(tuned_lgbm_trading_summary)

display(lgbm_baseline_trading_folds)
display(tuned_lgbm_trading_folds)

In [ ]:
# Modeling Step 25: LightGBM Feature Importance
# This is fitted on train_cv_data only for interpretation.
# It is not final test evaluation.
tuned_lgbm_pipe.fit(X_train_cv, y_train_cv)

lgbm_feature_importance = pd.DataFrame({
    "feature": selected_feature_cols,
    "importance": tuned_lgbm_pipe.named_steps["model"].model_.feature_importances_
}).sort_values("importance", ascending=False)

display(lgbm_feature_importance)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=lgbm_feature_importance.head(15).sort_values("importance"),
    x="importance",
    y="feature"
)
plt.title("Tuned LightGBM: Top 15 Feature Importances")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 8. Model selection and final holdout evaluation

Select the final specification from training/CV evidence only. Fit it once on the full training/CV period and evaluate it once on the protected chronological holdout.

In [ ]:
# Modeling Step 26: Compare All Candidate Models - Classification CV
# This comparison uses only train_cv_data through walk-forward CV.
# Final test_data is not used here.
classification_summary_list = [
    majority_summary,
    logistic_baseline_summary,
    tuned_logistic_summary,
    rf_baseline_summary,
    tuned_rf_summary,
    xgb_baseline_summary,
    tuned_xgb_summary,
    lgbm_baseline_summary,
    tuned_lgbm_summary
]

fold_summary_list = [
    majority_folds,
    logistic_baseline_folds,
    tuned_logistic_folds,
    rf_baseline_folds,
    tuned_rf_folds,
    xgb_baseline_folds,
    tuned_xgb_folds,
    lgbm_baseline_folds,
    tuned_lgbm_folds
]

all_model_cv_summary = pd.concat(
    classification_summary_list,
    axis=0
).reset_index(drop=True)

all_model_fold_comparison = pd.concat(
    fold_summary_list,
    axis=0
).reset_index(drop=True)

display(all_model_cv_summary)

all_model_cv_pivot = (
    all_model_cv_summary
    .pivot(index="model", columns="metric", values="cv_mean")
    .sort_values("balanced_accuracy", ascending=False)
)

display(all_model_cv_pivot)

display(all_model_fold_comparison)

plt.figure(figsize=(13, 5))
sns.barplot(
    data=all_model_cv_summary[
        all_model_cv_summary["metric"].isin(["accuracy", "balanced_accuracy", "macro_f1"])
    ],
    x="model",
    y="cv_mean",
    hue="metric"
)
plt.xticks(rotation=35, ha="right")
plt.title("Walk-Forward CV Classification Performance: All Candidate Models")
plt.ylabel("CV Mean Score")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
sns.lineplot(
    data=all_model_fold_comparison,
    x="fold",
    y="cv_balanced_accuracy",
    hue="model",
    marker="o"
)
plt.title("Fold-Level CV Balanced Accuracy: All Candidate Models")
plt.ylabel("CV Balanced Accuracy")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
sns.lineplot(
    data=all_model_fold_comparison,
    x="fold",
    y="cv_macro_f1",
    hue="model",
    marker="o"
)
plt.title("Fold-Level CV Macro F1: All Candidate Models")
plt.ylabel("CV Macro F1")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 27: Compare All Candidate Models - Trading CV
# This comparison uses validation-fold trading performance inside train_cv_data only.
# Final test_data is not used here.
trading_summary_list = [
    majority_trading_summary,
    logistic_baseline_trading_summary,
    tuned_logistic_trading_summary,
    rf_baseline_trading_summary,
    tuned_rf_trading_summary,
    xgb_baseline_trading_summary,
    tuned_xgb_trading_summary,
    lgbm_baseline_trading_summary,
    tuned_lgbm_trading_summary
]

trading_fold_list = [
    majority_trading_folds,
    logistic_baseline_trading_folds,
    tuned_logistic_trading_folds,
    rf_baseline_trading_folds,
    tuned_rf_trading_folds,
    xgb_baseline_trading_folds,
    tuned_xgb_trading_folds,
    lgbm_baseline_trading_folds,
    tuned_lgbm_trading_folds
]

all_model_trading_summary = pd.concat(
    trading_summary_list,
    axis=0
).reset_index(drop=True)

all_model_trading_folds = pd.concat(
    trading_fold_list,
    axis=0
).reset_index(drop=True)

display(all_model_trading_summary)
display(all_model_trading_folds)

plt.figure(figsize=(13, 5))
sns.barplot(
    data=all_model_trading_summary.sort_values("sharpe", ascending=False),
    x="model",
    y="sharpe"
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xticks(rotation=35, ha="right")
plt.title("Walk-Forward CV Trading Sharpe: All Candidate Models")
plt.ylabel("Sharpe")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
sns.barplot(
    data=all_model_trading_summary.sort_values("false_long_rate", ascending=True),
    x="model",
    y="false_long_rate"
)
plt.xticks(rotation=35, ha="right")
plt.title("Walk-Forward CV False Long Rate: All Candidate Models")
plt.ylabel("False Long Rate")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
sns.lineplot(
    data=all_model_trading_folds,
    x="fold",
    y="sharpe",
    hue="model",
    marker="o"
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Fold-Level Trading Sharpe: All Candidate Models")
plt.ylabel("Sharpe")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
sns.lineplot(
    data=all_model_trading_folds,
    x="fold",
    y="false_long_rate",
    hue="model",
    marker="o"
)
plt.title("Fold-Level False Long Rate: All Candidate Models")
plt.ylabel("False Long Rate")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
sns.lineplot(
    data=all_model_trading_folds,
    x="fold",
    y="active_trade_rate",
    hue="model",
    marker="o"
)
plt.title("Fold-Level Active Trade Rate: All Candidate Models")
plt.ylabel("Active Trade Rate")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 28: Core Tuned Model Comparison
# Compare only the strongest version of each model family.
core_model_names = [
    "Tuned Logistic Regression",
    "Tuned Random Forest",
    "Tuned XGBoost",
    "Tuned LightGBM"
]

core_cv_summary = all_model_cv_summary[
    all_model_cv_summary["model"].isin(core_model_names)
].copy()

core_trading_summary = all_model_trading_summary[
    all_model_trading_summary["model"].isin(core_model_names)
].copy()

core_fold_comparison = all_model_fold_comparison[
    all_model_fold_comparison["model"].isin(core_model_names)
].copy()

core_trading_folds = all_model_trading_folds[
    all_model_trading_folds["model"].isin(core_model_names)
].copy()

display(core_cv_summary)

core_cv_pivot = (
    core_cv_summary
    .pivot(index="model", columns="metric", values="cv_mean")
    .sort_values("balanced_accuracy", ascending=False)
)

display(core_cv_pivot)
display(core_trading_summary)

plt.figure(figsize=(11, 5))
sns.barplot(
    data=core_cv_summary[
        core_cv_summary["metric"].isin(["accuracy", "balanced_accuracy", "macro_f1"])
    ],
    x="model",
    y="cv_mean",
    hue="metric"
)
plt.xticks(rotation=30, ha="right")
plt.title("Core Tuned Models: Walk-Forward CV Classification Metrics")
plt.ylabel("CV Mean Score")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=core_fold_comparison,
    x="fold",
    y="cv_balanced_accuracy",
    hue="model",
    marker="o"
)
plt.title("Core Tuned Models: Fold-Level CV Balanced Accuracy")
plt.ylabel("CV Balanced Accuracy")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=core_fold_comparison,
    x="fold",
    y="cv_macro_f1",
    hue="model",
    marker="o"
)
plt.title("Core Tuned Models: Fold-Level CV Macro F1")
plt.ylabel("CV Macro F1")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

core_trading_long = core_trading_summary.melt(
    id_vars="model",
    value_vars=[
        "annualized_return",
        "annualized_volatility",
        "sharpe",
        "max_drawdown",
        "hit_rate_active_trades",
        "false_long_rate",
        "active_trade_rate"
    ],
    var_name="metric",
    value_name="value"
)

display(core_trading_long)

plt.figure(figsize=(11, 5))
sns.barplot(
    data=core_trading_long[
        core_trading_long["metric"].isin(["sharpe", "false_long_rate", "active_trade_rate"])
    ],
    x="model",
    y="value",
    hue="metric"
)
plt.xticks(rotation=30, ha="right")
plt.title("Core Tuned Models: Key Trading Metrics")
plt.ylabel("Metric Value")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=core_trading_folds,
    x="fold",
    y="sharpe",
    hue="model",
    marker="o"
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Core Tuned Models: Fold-Level Trading Sharpe")
plt.ylabel("Sharpe")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=core_trading_folds,
    x="fold",
    y="false_long_rate",
    hue="model",
    marker="o"
)
plt.title("Core Tuned Models: Fold-Level False Long Rate")
plt.ylabel("False Long Rate")
plt.xlabel("Walk-Forward Fold")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling Step 29: Final CV-Based Model Selection Table
# This table combines classification, trading, and overfitting information.
# It is based only on train/CV walk-forward results.
# Final test_data is still untouched.
classification_wide = all_model_cv_summary.pivot(
    index="model",
    columns="metric",
    values=["train_mean", "cv_mean", "cv_std"]
)

classification_wide.columns = [
    f"{a}_{b}" for a, b in classification_wide.columns
]

classification_wide = classification_wide.reset_index()

selection_table = classification_wide.merge(
    all_model_trading_summary,
    on="model",
    how="left"
)

selection_table["balanced_accuracy_gap"] = (
    selection_table["train_mean_balanced_accuracy"]
    - selection_table["cv_mean_balanced_accuracy"]
)

selection_table["macro_f1_gap"] = (
    selection_table["train_mean_macro_f1"]
    - selection_table["cv_mean_macro_f1"]
)

selection_display_cols = [
    "model",
    "cv_mean_accuracy",
    "cv_mean_balanced_accuracy",
    "cv_std_balanced_accuracy",
    "cv_mean_macro_f1",
    "cv_std_macro_f1",
    "balanced_accuracy_gap",
    "macro_f1_gap",
    "annualized_return",
    "annualized_volatility",
    "sharpe",
    "max_drawdown",
    "hit_rate_active_trades",
    "false_long_rate",
    "active_trade_rate",
    "short_pred_share",
    "flat_pred_share",
    "long_pred_share"
]

selection_table = (
    selection_table[selection_display_cols]
    .sort_values(
        by=["cv_mean_balanced_accuracy", "cv_mean_macro_f1", "sharpe"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

display(selection_table)

core_selection_table = selection_table[
    selection_table["model"].isin(core_model_names)
].copy()

display(core_selection_table)

In [ ]:
# Modeling Step 30: Select Final Model Based on CV Evidence
# Final model selection is based only on walk-forward CV results.
# No final test result has been used for this decision.
final_model_name = "Tuned LightGBM"
final_model = tuned_lgbm_pipe

print("Final model selected based on walk-forward CV evidence:")
print(final_model_name)

final_model_cv_row = selection_table[
    selection_table["model"] == final_model_name
].copy()

display(final_model_cv_row)

In [ ]:
# Modeling Step 31: Final Test Evaluation
# This is the only final test evaluation.
# The final model has already been selected using train/CV results only.
final_model.fit(X_train_cv, y_train_cv)

y_test_pred = final_model.predict(X_test)

classification_summary, confusion_df = evaluate_classification(
    y_true=y_test,
    y_pred=y_test_pred,
    model_name=final_model_name
)

trading_summary, test_eval_df = evaluate_trading_performance(
    test_data=test_data,
    y_true=y_test,
    y_pred=y_test_pred,
    model_name=final_model_name
)

print("Final Test Classification Summary:")
display(pd.DataFrame([classification_summary]))

print("Final Test Confusion Matrix:")
display(confusion_df)

print("Final Test Trading Summary:")
display(trading_summary)

In [ ]:
# Modeling Step 32: Final Test Trading Visualization
test_eval_df = test_eval_df.copy()

test_eval_df["strategy_cum_ret"] = test_eval_df["strategy_ret"].cumsum()
test_eval_df["spy_cum_next_ret"] = test_eval_df["next_ret"].cumsum()

plt.figure(figsize=(12, 5))
plt.plot(test_eval_df.index, test_eval_df["strategy_cum_ret"], label=f"{final_model_name} Strategy")
plt.plot(test_eval_df.index, test_eval_df["spy_cum_next_ret"], label="Buy-and-Hold SPY Next Return")
plt.axhline(0, linestyle="--", linewidth=1)
plt.title(f"Final Test Cumulative Return: {final_model_name}")
plt.ylabel("Cumulative Log Return")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.countplot(
    x=pd.Series(y_test_pred).map({-1: "Short", 0: "Flat", 1: "Long"}),
    order=["Short", "Flat", "Long"]
)
plt.title(f"Final Test Signal Distribution: {final_model_name}")
plt.xlabel("Predicted Signal")
plt.ylabel("Count")
plt.tight_layout()
plt.show()